# 0. Import

In [ ]:
import argparse
import pyprind

import numpy as np
import pandas as pd

from scipy.spatial.distance import cdist
from scipy.interpolate import interp1d
from scipy import stats

from fancyimpute import KNN

from collections import Counter
from tqdm import tqdm

import matplotlib.pyplot as plt
%matplotlib inline

# 0. 필요한 function 정리

In [ ]:
def SAH(input, vitalslab_hold, adjust=0):
    '''Matthieu Komorowski - Imperial College London 2017 
    will copy a value in the rows below if the missing values are within the
    hold period for this variable (e.g. 48h for weight, 2h for HR...)
    vitalslab_hold = 2x55 cell (with row1 = strings of names ; row 2 = hold time)'''
    temp = np.copy(input)
    hold = vitalslab_hold.values[0, :]
    nrow, ncol = temp.shape

    lastcharttime = np.zeros(ncol)
    lastvalue = np.zeros(ncol)
    oldstayid = temp[0, 1]

    bar_SAH = pyprind.ProgBar(ncol-(3+adjust))
    for i in range(3+adjust,ncol):
        bar_SAH.update()
        for j in range(nrow):
            if oldstayid != temp[j, 1]:
                lastcharttime = np.zeros(ncol)
                lastvalue = np.zeros(ncol)
                oldstayid = temp[j, 1]
            if not np.isnan(temp[j, i]):
                lastcharttime[i] = temp[j, 2]
                lastvalue[i] = temp[j, i]
            if j > 0:
                if (np.isnan(temp[j, i])) and (temp[j, 1] == oldstayid) and ((temp[j, 2] - lastcharttime[i]) <= hold[i-(3+adjust)]*3600):
                    temp[j,i] = lastvalue[i]
    return temp

def fixgaps(x):
    '''FIXGAPS Linearly interpolates gaps in a time series
    YOUT=FIXGAPS(YIN) linearly interpolates over NaN
    in the input time series (may be complex), but ignores
    trailing and leading NaN.
    R. Pawlowicz 6/Nov/99'''
    y = np.copy(x)
    bd = np.isnan(x)
    gd = np.arange(len(x))[~bd]
    bd[:min(gd)] = False
    bd[max(gd)+1:] = False
    y[bd] = interp1d(gd,x[gd])(np.arange(len(x))[bd])
    return y

def deloutabove(a, col_no, a_max):
    a[a[:,col_no] > a_max, col_no] = np.nan 
    return a

def deloutbelow(a, col_no, a_min):
    a[a[:,col_no] < a_min, col_no] = np.nan 
    return a

# 1. Data load

1. sepsis 처리 과정 참고하여, 사용할 수 있는 데이터들은 모두 추출.
    - *demog* : 전체 환자에 관한 admittime, intime, age, dod, dob, elixhauser score등을 수집한 것이므로 morta_hosp, morta_90과 같이 불필요한 column은 제거하고 사용 가능.

In [ ]:
demog   = pd.read_csv('./demog.csv', sep = ',')
labU    = [pd.read_csv('./labs_ce.csv', sep = ',') , pd.read_csv('./labs_le.csv', sep = ',')]
ce010   = pd.read_csv('./ce010000.csv',      sep = ',')
ce1020  = pd.read_csv('./ce1000020000.csv',  sep = ',')
ce2030  = pd.read_csv('./ce2000030000.csv',  sep = ',')
ce3040  = pd.read_csv('./ce3000040000.csv',  sep = ',')
ce4050  = pd.read_csv('./ce4000050000.csv',  sep = ',')
ce5060  = pd.read_csv('./ce5000060000.csv',  sep = ',')
ce6070  = pd.read_csv('./ce6000070000.csv',  sep = ',')
ce7080  = pd.read_csv('./ce7000080000.csv',  sep = ',')
ce8090  = pd.read_csv('./ce8000090000.csv',  sep = ',')
ce90100 = pd.read_csv('./ce90000100000.csv', sep = ',')
###################################################
hepa_cv = pd.read_csv("./heparin_cv.csv", sep=',')  # Some icustay_id values are null.
hepa_mv = pd.read_csv("./heparin_mv.csv", sep=',')  # Some icustay_id values are null.
# hepa_cv = pd.read_csv("./heparin_cv_check_IV.csv", sep=',')  # Some icustay_id values are null.
# hepa_mv = pd.read_csv("./heparin_mv_check_IV.csv", sep=',')  # Some icustay_id values are null.
###################################################
vasoMV  = pd.read_csv('./vaso_mv.csv',       sep = ',')
vasoCV  = pd.read_csv('./vaso_cv.csv',       sep = ',')

UOpreadm = pd.read_csv('./preadm_uo.csv',     sep = ',')
UO       = pd.read_csv('./uo.csv',            sep = ',')

# 2. 초기 데이터 세팅

In [ ]:
labU[1].rename(columns = {'timestp': 'charttime'}, inplace=True) 
labU = pd.concat(labU, sort=False, ignore_index=True)

demog['morta_90'].fillna(0, inplace=True) # This step is not needed here, but retain it to follow the original workflow.
demog['morta_hosp'].fillna(0, inplace=True) # This step is not needed here, but retain it to follow the original workflow.
demog['elixhauser'].fillna(0, inplace=True)

demog = demog.drop_duplicates(subset=['admittime','dischtime'],keep='first')

icustayidlist = list(demog.icustay_id.values)

# 3. readmission 여부 계산

    - 퇴원 후 30일 기준

In [ ]:
# Calculate the accurate readmission using the demographics data 
# (the SQL code from Komorowski, et al incorrectly cumulatively counts how many icu stays each patient has (preprocess.py:line 414) 
# and does a coarse boolean check if this number is >1). A readmission is now correctly defined by 
# whether the patient has returned to the ICU within 30 days of being previously discharged.

# This is done by grouping all the discharge times for each patient and using them in a comparison 
# with the current row's admission time to see if it's within the 30 day cutoff
subj_dischtime_list = demog.sort_values(by='admittime').groupby('subject_id').apply(lambda df: np.unique(df.dischtime.values)) # Create list of discharge times for each patient (output is a dict keyed by 'subject_id')

def determine_readmission(s, dischtimes=subj_dischtime_list,cutoff=3600*24*30):
    '''
    determine_readmisson evaluates each row of the provided dataframe (designed to operate on the demographics table)
    and chooses whether the current admission occurs within the cutoff of the previous discharge 
    (here, cutoff=30 days is the default)
    '''
    subject, admission, discharge = s[['subject_id','admittime','dischtime']]
    
    # Check for readmission
    subj_stay_idx = np.where(dischtimes[subject]==discharge)[0][0]
    s['re_admission'] = 0
    if subj_stay_idx > 0:
        if (admission - dischtimes[subject][subj_stay_idx-1]) <= cutoff:
            s['re_admission'] = 1
            
    return s
# Apply the above function to determine the appropriate readmissions
demog = demog.apply(determine_readmission,axis=1)

# 4. Fill-in missing ICUSTAY IDs in heparin_cv!!

    - demog 데이터를 이용해서 hepa_cv df의 비어있는 icustay_id 채우기
    - 처리한 후에도 null값 인 것은 그냥 지우기

In [ ]:
print('Filling-in missing ICUSTAY IDs in hepa_cv')
for i in tqdm(hepa_cv.index.tolist()):
    if np.isnan(hepa_cv.loc[i, 'icustay_id']):
        o         = hepa_cv.loc[i, 'charttime'] 
        subjectid = hepa_cv.loc[i, 'subject_id']
        hadmid    = hepa_cv.loc[i, 'hadm_id']
        ii        = demog.index[demog['subject_id'] == subjectid].tolist()
        jj        = demog.index[(demog['subject_id'] == subjectid) & (demog['hadm_id'] == hadmid)].tolist()
        for j in range(len(ii)):
            if (o >= demog.loc[ii[j], 'intime'] - 48*3600) and (o <= demog.loc[ii[j], 'outtime'] + 48*3600):
                hepa_cv.loc[i,'icustay_id'] = demog.loc[ii[j], 'icustay_id']
            elif len(ii)==1:   # If we cant confirm from admission and discharge time but there is only 1 admission: it's the one!!
                hepa_cv.loc[i,'icustay_id'] = demog.loc[ii[j], 'icustay_id']
                
print('Filling-in missing ICUSTAY IDs in hepa_cv - 2')                
for i in tqdm(hepa_cv.index.tolist()):
    if np.isnan(hepa_cv.loc[i, 'icustay_id']):
        subjectid = hepa_cv.loc[i, 'subject_id']
        hadmid    = hepa_cv.loc[i, 'hadm_id']
        jj        = demog.index[(demog['subject_id'] == subjectid) & (demog['hadm_id'] == hadmid)].tolist()
        if len(jj) == 1:
            hepa_cv.loc[i,'icustay_id'] = demog.loc[jj[0], 'icustay_id']

In [ ]:
# Remove rows whose null values remain after processing because matching cannot resolve them.
hepa_cv = hepa_cv[~hepa_cv['icustay_id'].isna()]
hepa_mv = hepa_mv[~hepa_mv['icustay_id'].isna()]

# 6. heparin 단위 통일 : unit

- hepa_mv는 모두 'unit' 데이터 이고 hepa_cv 단위는 'unit'과 'ml'가 있음.
- 'unit'데이터가 가장 많고 'ml'는 식염수는 소금산 같은 다른 약물이 섞여있어서 헤파린을 정확하게 표현하는 unit단위를 사용하고자함.
- ml단위를 가지는 icustay_id의 처방기록을 모두 살펴보니, 대부분 10units/ml로 헤파린이 처방되었고, icustay_id가 [203154, 293629]인 경우만 '25,000 unit Premix Bag'라는 이름으로 처방됨. ml를 설명하는 부분이 없으므로 unit으로 변환 불가함.


- so, 결론적으로 ml->unit 변환은 sql query로 일괄적으로 실행하였고,
- icustay_id가 [203154, 293629] 면서 amountuom이 'ml'인 데이터만 여기서 지우자.(그리고 내가볼 때 정상적인 데이터 사이에 끼어있는것으로 봐서 잘못 입력된듯)

In [ ]:
rm_1 = hepa_cv.loc[(hepa_cv['icustay_id']==203154)&(hepa_cv['amountuom']=='ml')].index
hepa_cv.drop(rm_1, inplace=True)

rm_2 = hepa_cv.loc[(hepa_cv['icustay_id']==293629)&(hepa_cv['amountuom']=='ml')].index
hepa_cv.drop(rm_2, inplace=True)

# 7. 실시간 약물 투여 정보에 대한 normalized rate 계산

- norm_rate_of_infusion = tev(total equivalent volume) * rate / amount
- rate 값이 있으면 infusion, 없으면 bolus라고 가정함. (sepsis의 가정)

In [ ]:
# if 100 ml of hypertonic fluid (600 mosm/l) is given at 100 ml/h (given in 1h) it is 200 ml of NS equivalent
# so the normalized rate of infusion is 200 ml/h (different volume in same duration)
hepa_mv['norm_rate_of_infusion'] = hepa_mv['tev']*hepa_mv['rate']/hepa_mv['amount']

# 8. icustay_id 별로 첫 번째 헤파린 투여 시각 데이터 추출

    - sepsis에서 sepsis '감염 추정 시기'를 기준 시간으로 사용한 것과 같이 여기서는 '첫 번째 헤파린 투여 시간'을 기준으로 사용한다.

In [ ]:
onset = dict()
num_onset = 0

input_data = dict()

for icustayid in tqdm(icustayidlist):
    onset[icustayid] = np.zeros(3)
    
    mv = hepa_mv.loc[hepa_mv['icustay_id']==icustayid,'starttime']
    cv = hepa_cv.loc[hepa_cv['icustay_id']==icustayid,'charttime']
    subj_cv = hepa_cv.loc[hepa_cv['icustay_id']==icustayid,'subject_id']
    subj_mv = hepa_mv.loc[hepa_mv['icustay_id']==icustayid,'subject_id']
    
    chart_time = list(mv)+list(cv)
    if len(chart_time)>0:

        chart_time.sort()        
        input_data[icustayid] = len(chart_time)
        
        if not subj_cv.empty:
            subj = subj_cv.iloc[0]
        else:
            subj = subj_mv.iloc[0]

        
        onset[icustayid][0]=subj
        onset[icustayid][1]=icustayid
        onset[icustayid][2]=chart_time[0]
        num_onset+=1

In [ ]:
## Remove subject_id, hadm_id, and amountuom because they interfere with subsequent processing.
hepa_cv.drop(columns=['subject_id','hadm_id','amountuom'], inplace=True)
hepa_mv.drop(columns=['subject_id'],inplace=True)

# 9. Reference table 맵핑

- itemid를 임의의 작은 숫자로 변환시킴.
- itemid는 다른데 동일한 종류의 약물은 같은 id로 맵핑시켜줌
- 나중에 itemid의 명칭을 모를 때 여기로 돌아와서 mapping 시켜서 찾으면 됨. (check_itemid 폴더에 전/후 결과 있음)

In [ ]:
# Replacing item_ids with column numbers from reference tables
print('Full ICU -- Replacing item_ids with column numbers from reference tables')

# Replace itemid in labs with column number
# This will accelerate process later
Reflabs = pd.read_csv("ReferenceFiles/Reflabs.tsv", sep = '\t', header=None)
Reflabs_values = np.unique(Reflabs.fillna(-10000))[1:]
Reflabs_id_dict = {}
for r in Reflabs_values:
    try:
        Reflabs_id_dict[r] = np.max(np.where(Reflabs.values == r)[0]) + 1 # for row: +1 due to Index correction python        
    except:
        print(r)
        break
itemid_col = labU.columns.tolist().index('itemid')
labU_temp = labU.values
for c in range(labU_temp.shape[0]):
    labU_temp[c,itemid_col] = Reflabs_id_dict[labU_temp[c,itemid_col]]
for i, c in enumerate(labU.columns.tolist()):
    labU.loc[:,c] = labU_temp[:,i]


# Replace itemid in vitals with col number
Refvitals = pd.read_csv("ReferenceFiles/Refvitals.tsv", sep = '\t', header=None)
Refvitals_values = np.unique(Refvitals.fillna(-10000))[1:]
Refvitals_id_dict = {}
for r in Refvitals_values:
    Refvitals_id_dict[r] = np.max(np.where(Refvitals.values == r)[0]) + 1 # +1 due to index correction for Python from MATLAB
ce_dfs = [ce010, ce1020, ce2030, ce3040, ce4050, ce5060, ce6070, ce7080, ce8090, ce90100]
for ce_df in ce_dfs:
    itemid_col = ce_df.columns.tolist().index('itemid')
    ce_df_temp = ce_df.values
    for c in range(ce_df_temp.shape[0]):
        ce_df_temp[c,itemid_col] = Refvitals_id_dict[ce_df_temp[c,itemid_col]]
    for i, c in enumerate(ce_df.columns.tolist()):
        ce_df.loc[:,c] = ce_df_temp[:,i]

# 10. Lab test 결과에 관련된 데이터를 첫 번째 heparin 투약 시간이 존재하는 icustay_id기준으로 통합시킴.

- heparin은 sepsis의 감염추정시기와 같은 기준 시점으로 첫 번째 heparin 투약 시점을 사용함.
- 18세 이상의 환자들만 취급함.
- 첫 투약 시점으로부터 사용가능한 모든 데이터 사용
- sepsis와 달리 Mechanical ventilation 데이터는 사용하지 않아서 뒤에 66,67번째 column은 빈값임.

In [ ]:
print(' Full ICU --  Making an array with all unique charttime (1 per row) and all items in columns.')
reformat = np.nan*np.ones((2000000,68))  # Final table 
qstime = dict()
winb4 = 0   # Lower limit for inclusion of data
winaft = 48  # Upper limit (96h after)
irow = 0  # Recording row for summary table

for icustayid in tqdm(icustayidlist):
    qstime[icustayid] = np.zeros(4)
    qst = onset[icustayid][2] # Flag the first heparin administration.
    if qst > 0:  # if we have a flag
        d1 = demog.loc[demog['icustay_id'] == icustayid, ['age', 'dischtime']].values[0] # Age of patient + discharge time
        if d1[0] > 6574:  # If older than 18 years old : 6574days
            # CHARTEVENTS
            if (icustayid-200000) < 10000:
                temp=ce010
            elif (icustayid-200000) < 20000:
                temp=ce1020
            elif (icustayid-200000) < 30000:
                temp=ce2030
            elif (icustayid-200000) < 40000:
                temp=ce3040
            elif (icustayid-200000) < 50000:
                temp=ce4050
            elif (icustayid-200000) < 60000:
                temp=ce5060
            elif (icustayid-200000) < 70000:
                temp=ce6070
            elif (icustayid-200000) < 80000:
                temp=ce7080
            elif (icustayid-200000) < 90000:
                temp=ce8090
            else:
                temp=ce90100
            
            # CHARTEVENTS
            temp = temp[temp['icustay_id'] == icustayid] 

            ii = (temp['charttime'] >= qst) & (temp['charttime'] <= qst + (winaft)*3600) 
            temp = temp.loc[ii]   # Only time period of interest
            
            # LABEVENTS
            ii = labU['icustay_id'] == icustayid
            temp2 = labU.loc[ii]
            ii = (temp2['charttime'] >= qst) & (temp2['charttime'] <= qst + (winaft)*3600) 
            temp2 = temp2.loc[ii]   # Only time period of interest
            
            t = np.unique(pd.concat([temp['charttime'], temp2['charttime']], ignore_index=True).values) # List of unique timestamps from all 3 sources / sorted in ascending order

            if len(t) > 0:
                for i in range(len(t)):
                    #CHARTEVENTS
                    ii = temp['charttime'] == t[i]
                    col = temp.loc[ii,'itemid']
                    value = temp.loc[ii,'valuenum']
                    reformat[irow, 0] = i+1 #timestep  
                    reformat[irow, 1] = icustayid
                    reformat[irow, 2] = t[i] #charttime
                    reformat[irow, 2+col.astype(int).values] = value.values # Store available values; each itemid is automatically assigned to its column.
                    
                    
                    # LAB VALUES
                    ii = temp2['charttime'] == t[i]
                    col = temp2.loc[ii, 'itemid']
                    value = temp2.loc[ii, 'valuenum']
                    reformat[irow, 30+col.astype(int).values] = value.values  # Store available values
                    
                    irow += 1
                
                qstime[icustayid][0] = qst # Flag for presumed infection
                # SAVE FIRST and LAST TIMESTAMPS, in QSTIME, for each ICUSTAYID
                qstime[icustayid][1] = t[0]   # First timestamp
                qstime[icustayid][2] = t[-1]  # Last timestamp
                qstime[icustayid][3] = d1[1]  # Discharge time
                
reformat = np.delete(reformat, range(irow, len(reformat)) ,axis=0)  # Delete unused rows

In [ ]:
len(pd.Series(reformat[:,1]).unique())

# 11. Outlier 처리

- 각 column 순서는 sepsis와 동일하므로, outlier 처리도 동일하게 follow up!!

In [ ]:
print('Full ICU -- Handling outliers')

# Weight
reformat = deloutabove(reformat, 4, 300) 

# Heart Rate
reformat = deloutabove(reformat, 7, 250)

# Blood Pressure
reformat = deloutabove(reformat, 8, 300)
reformat = deloutbelow(reformat, 9, 0) 
reformat = deloutabove(reformat, 9, 200)
reformat = deloutbelow(reformat, 10, 0) 
reformat = deloutabove(reformat, 10, 200) 

# Respiratory Rate
reformat = deloutabove(reformat, 11, 80) 

# SpO2
reformat = deloutabove(reformat, 12, 150) 
reformat[reformat[:, 12]>100, 12] = 100

# Temperature
reformat[(reformat[:, 13] > 90) & (np.isnan(reformat[:, 14])), 14] = reformat[(reformat[:, 13] > 90) & (np.isnan(reformat[:, 14])), 13]
reformat = deloutabove(reformat, 13, 90) 

# Interface / is in col 22
# FiO2
reformat = deloutabove(reformat, 22, 100) 
reformat[reformat[:, 22] < 1 , 22] = reformat[reformat[:,22] < 1 , 22]*100

reformat = deloutbelow(reformat, 22, 20) 
reformat = deloutabove(reformat, 23, 1.5)

# O2 FLOW
reformat = deloutabove(reformat, 24, 70)

#PEEP
reformat=deloutbelow(reformat, 25, 0) 
reformat=deloutabove(reformat, 25, 40) 

# Total Volume
reformat=deloutabove(reformat, 26, 1800)

# Mean Volume
reformat=deloutabove(reformat, 27, 50)

# Potassium
reformat=deloutbelow(reformat, 31, 1) 
reformat=deloutabove(reformat, 31, 15) 

# Sodium
reformat=deloutbelow(reformat, 32, 95) 
reformat=deloutabove(reformat, 32, 178)

# Chloride
reformat=deloutbelow(reformat, 33, 70) 
reformat=deloutabove(reformat, 33, 150)

# Glucose
reformat=deloutbelow(reformat, 34, 1) 
reformat=deloutabove(reformat, 34, 1000)

# Creatinine
reformat=deloutabove(reformat, 36, 150)

# Magnesium
reformat=deloutabove(reformat, 37, 10)

# Calcium
reformat=deloutabove(reformat, 38, 20)

# Ionized Calcium
reformat=deloutabove(reformat, 39, 5) 

# CO2
reformat=deloutabove(reformat, 40, 120) 

# SGPT/SGOT
reformat=deloutabove(reformat, 41, 10000) 
reformat=deloutabove(reformat, 42, 10000) 

# Hb/Ht
reformat=deloutabove(reformat, 49, 20) 
reformat=deloutabove(reformat, 50, 65) 

# White Blood Cells
reformat=deloutabove(reformat, 52, 500) 

# Platelets
reformat=deloutabove(reformat, 53, 2000)

# INR
reformat=deloutabove(reformat, 57, 20) 

# pH
reformat=deloutbelow(reformat, 58, 6.7) 
reformat=deloutabove(reformat, 58, 8) 

# pO2
reformat=deloutabove(reformat, 59, 700)

# pCO2
reformat=deloutabove(reformat, 60, 200)

# Base Excess
reformat=deloutbelow(reformat, 61, -50)

# Lactate
reformat=deloutabove(reformat, 62, 30) 

# 12. more data manipulation / imputation from existing values

- 기존에 존재하는 데이터들과의 관계를 이용해서 빠져있는 데이터를 채움.
        ex. RASS 데이터를 이용해서 GCS를 추정.
    
- FIO2 계산하려고 sample-and-hold interpolation 수행. 나머지에는 적용안됨.
    
- 각 column 순서는 sepsis와 동일하므로, outlier 처리도 동일하게 follow up!!

In [ ]:
# Estimate GCS from RASS - data from Wesley JAMA 2003
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] >= 0), 5] = 15
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -1), 5] = 14
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -2), 5] = 12
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -3), 5] = 11
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -4), 5] = 6
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -5), 5] = 3

# FiO2
reformat[(~np.isnan(reformat[:, 22])) & (np.isnan(reformat[:, 23])), 23] = reformat[(~np.isnan(reformat[:, 22])) & (np.isnan(reformat[:, 23])), 22] / 100
reformat[(~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 22])), 22] = reformat[(~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 22])), 23] * 100

print('Full ICU -- Doing sample and hold')
sample_and_hold = pd.read_csv('ReferenceFiles/sample_and_hold.csv', index_col = None)

reformatsah = SAH(reformat,sample_and_hold)  # Do SAH first to handle this task

# NO FiO2, YES O2 flow, no interface OR cannula
ii = np.where((np.isnan(reformatsah[:, 22])) & (~np.isnan(reformatsah[:, 24])) & ((reformatsah[:, 21] == 0) | (reformatsah[:, 21] == 2)))[0] #As np.where returns a tuple
reformat[ii[reformatsah[ii, 24] <= 15], 22] = 70
reformat[ii[reformatsah[ii, 24] <= 12], 22] = 62
reformat[ii[reformatsah[ii, 24] <= 10], 22] = 55
reformat[ii[reformatsah[ii, 24] <= 8], 22]  = 50
reformat[ii[reformatsah[ii, 24] <= 6], 22]  = 44
reformat[ii[reformatsah[ii, 24] <= 5], 22]  = 40
reformat[ii[reformatsah[ii, 24] <= 4], 22]  = 36
reformat[ii[reformatsah[ii, 24] <= 3], 22]  = 32
reformat[ii[reformatsah[ii, 24] <= 2], 22]  = 28
reformat[ii[reformatsah[ii, 24] <= 1], 22]  = 24

# NO FiO2, NO O2 flow, no interface OR cannula
ii = np.where((np.isnan(reformatsah[:, 22])) & np.isnan(reformatsah[:, 24]) & ((reformatsah[:, 21] == 0) | (reformatsah[:, 21] == 2)))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 22] = 21

# NO FiO2, YES O2 flow, face mask OR.... OR ventilator (assume it's face mask)
ii = np.where((np.isnan(reformatsah[:, 22])) & (~np.isnan(reformatsah[:, 24])) & 
((reformatsah[:, 21] == 1) | (reformatsah[:, 21]==3) | (reformatsah[:, 21] == 4) | (reformatsah[:, 21] == 5) | (reformatsah[:, 21]==6) | (reformatsah[:, 21]==9) | (reformatsah[:, 21]==10)))[0]
reformat[ii[reformatsah[ii, 24]<=15], 22] = 75
reformat[ii[reformatsah[ii, 24]<=12], 22] = 69
reformat[ii[reformatsah[ii, 24]<=10], 22] = 66
reformat[ii[reformatsah[ii, 24]<=8], 22]  = 58
reformat[ii[reformatsah[ii, 24]<=6], 22]  = 40
reformat[ii[reformatsah[ii, 24]<=4], 22]  = 36

# NO FiO2, NO O2 flow, face mask OR ....OR ventilator
ii = np.where(np.isnan(reformatsah[:, 22]) & np.isnan(reformatsah[:, 24]) & ((reformatsah[:, 21] == 1) | (reformatsah[:, 21] == 3) | 
(reformatsah[:, 21] == 4) | (reformatsah[:, 21] == 5) | (reformatsah[:, 21] == 6) | (reformatsah[:, 21] == 9) | (reformatsah[:, 21] == 10)))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 22] = np.nan

# NO FiO2, YES O2 flow, Non rebreather mask
ii = np.where(np.isnan(reformatsah[:, 22]) & (~np.isnan(reformatsah[:, 24])) & (reformatsah[:, 21] == 7))[0]
reformat[ii[reformatsah[ii, 24] >= 10], 22] = 90
reformat[ii[reformatsah[ii, 24] >= 15], 22] = 100
reformat[ii[reformatsah[ii, 24] < 10], 22]  = 80
reformat[ii[reformatsah[ii, 24] <= 8], 22]  = 70
reformat[ii[reformatsah[ii, 24] <= 6], 22]  = 60

# NO FiO2, NO O2 flow, NRM
ii= np.where(np.isnan(reformatsah[:, 22]) & np.isnan(reformatsah[:, 24]) & (reformatsah[:, 21]==7))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 22] = np.nan

# Update FiO2 columns again
ii = (~np.isnan(reformat[:, 22])) & (np.isnan(reformat[:,23]))
reformat[ii, 23] = reformat[ii, 22]/100
ii = (~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 22]))
reformat[ii, 22] = reformat[ii, 23]*100

# Blood Pressure
ii = (~np.isnan(reformat[:, 8])) & (~np.isnan(reformat[:, 9])) & np.isnan(reformat[:, 10])
reformat[ii, 10] = (3*reformat[ii, 9] - reformat[ii, 8])/2
ii = (~np.isnan(reformat[:, 8])) & (~np.isnan(reformat[:, 10])) & np.isnan(reformat[:, 9])
reformat[ii, 9] = (reformat[ii, 8] + 2*reformat[ii, 10])/3
ii = (~np.isnan(reformat[:, 9])) & (~np.isnan(reformat[:, 10])) & np.isnan(reformat[:, 8])
reformat[ii, 8] = 3*reformat[ii, 9] - 2*reformat[ii, 10]

# Temperature
# Some values recorded in the wrong column
ii = (reformat[:, 14] > 25) & (reformat[:, 14] < 45) # tempF close to 37deg??!
reformat[ii, 13] = reformat[ii, 14]
reformat[ii, 14] = np.nan
ii = reformat[:, 13] >70  # tempC > 70, likely recorded in Farenheit
reformat[ii, 14] = reformat[ii, 13]
reformat[ii, 13] = np.nan
ii = (~np.isnan(reformat[:, 13])) & np.isnan(reformat[:, 14])
reformat[ii, 14] = reformat[ii, 13]*1.8+32
ii = (~np.isnan(reformat[:, 14])) & np.isnan(reformat[:, 13])
reformat[ii, 13] = (reformat[ii, 14] - 32)/1.8

# Hb/Ht
ii = (~np.isnan(reformat[:,49])) & np.isnan(reformat[:, 50])
reformat[ii, 50] = (reformat[ii, 49] * 2.862) + 1.216
ii = (~np.isnan(reformat[:, 50])) & np.isnan(reformat[:, 49])
reformat[ii, 49] = (reformat[ii, 50] - 1.216)/2.862

# Bilirubin
ii = (~np.isnan(reformat[:, 43])) & np.isnan(reformat[:, 44])
reformat[ii, 44] = (reformat[ii, 43]*0.6934)-0.1752
ii = (~np.isnan(reformat[:, 44])) & np.isnan(reformat[:, 43])
reformat[ii, 43] = (reformat[ii, 44] + 0.1752)/0.6934



In [ ]:
np.save("./corpus/1hourly/reformat",reformat)

In [ ]:
reformat = np.load("./corpus/1hourly/reformat.npy")

In [ ]:
len(pd.Series(reformat[:,1]).unique())

# 13. SAH 수행. 

- PTT의 hold 범위를 0으로 지정하였기 때문에 PTT에 대해서는 interpolation 수행 안됨.
- PTT를 reward로 사용할 것이기 때문에 일단 그대로의 데이터가 있는것이 좋을 것 같아서 SAH 수행 안함. 
- 뒤에서 state로 넣어놓기는 할것인데 거기에는 그냥 Linear, KNN 적용해서 채울 것임.
- reward PTT에는 interpolation 적용 안할꺼임.

In [ ]:
########################################################################
#                      SAMPLE AND HOLD on RAW DATA
########################################################################
print('Full ICU -- SAMPLE AND HOLD on RAW DATA')
reformat = SAH(reformat[:,0:68],sample_and_hold)

# 13. 기존에 Lab test정보를 통합했던 reformat(df)에 투여기록을 추가로 통합.
    
- 기존의 lab test결과를 통합한 reformat 데이터프레임을 기준으로 함.
- heparin_mv, heparin_cv, vaso_cv, vaso_mv, urine의 투여기록을 통합시킴.
- vaso, urine은 나중에 SOFA score 계산하는데 필요해서 사용함.
- sepsis에서는 preadmission 기록을 통합하는데 우리는 preadmission 기록은 다루지 않음.(totvol)
- reformat2에 저장함.
- timestep : 1h

In [ ]:
print('Full ICU -- Data combination')

timestep = 1  # Resolution of timesteps, in hours
irow = 0
icustayidlist = np.unique(reformat[:,1]).astype(np.int32)
reformat2 = np.nan*np.ones((reformat.shape[0], 85))  # Output array  

num_patients = len(icustayidlist)  # Number of patients
# Adding 2 empty cols for future shock index=HR/SBP and P/F
reformat = np.hstack([reformat, np.nan*np.ones((reformat.shape[0], 2))])

for i in tqdm(range(len(icustayidlist))):
    icustayid = icustayidlist[i] 
    
    #CHARTEVENTS AND LAB VALUES
    temp = reformat[reformat[:,1] == icustayid,:]   # subtable of interest
    beg = temp[0,2]   # timestamp of first record
    
    #heparin FLUID STUFF
    iv = hepa_mv['icustay_id'] == icustayid         # rows of interest in hepa_mv
    input1 = hepa_mv[iv]                             # subset of interest
    iv = hepa_cv['icustay_id'] == icustayid         # rows of interest in hepa_cv
    input2 = hepa_cv[iv]                            # subset of interest
    startt = input1['starttime']                     # start of all infusions and boluses
    endt = input1['endtime']                         # end of all infusions and boluses
    rate = input1['norm_rate_of_infusion']    
    
    input1 = input1.values
    input2 = input2.values
    
    
    # Compute volume of fluid given before start of record!!!
    t0 = 0
    t1 = beg
    # Input from MetaVision (4 ways to compute)
    infu = np.nansum(rate*(endt-startt)*((endt<=t1) & (startt >= t0))/3600 + rate*(endt-t0)*((startt <= t0) & (endt <= t1) & (endt >= t0))/3600 +
                            rate*(t1-startt)*((startt >= t0) & (endt >=t1) & (startt <= t1))/3600 + rate*(t1-t0)*((endt >= t1) & (startt <= t0))/3600)
    # All boluses received during this timestep, from hepa_mv (need to check rate is NaN) and hepa_cv (simpler):
    bolus = np.nansum(input1[np.isnan(input1[:, 5]) & (input1[:, 1] >= t0) & (input1[:, 1] <= t1), 6]) + np.nansum(input2[(input2[:, 1] >= t0) & (input2[:, 1] <= t1), 4])
    totvol = np.nansum([infu,bolus])
    
    
    #########################################################################################
    #VASOPRESSORS    
    iv = vasoMV['icustay_id'] == icustayid  # rows of interest in vasoMV
    vaso1 = vasoMV[iv]    # subset of interest
    iv = vasoCV['icustay_id'] == icustayid   # rows of interest in vasoCV
    vaso2 = vasoCV[iv]    # subset of interest
    startv = vaso1['starttime'].values  # start of VP infusion
    endv = vaso1['endtime'].values      # end of VP infusions
    ratev = vaso1['rate_std'].values  # rate of VP infusion
    
    # DEMOGRAPHICS / gender, age, elixhauser, re-admit, died in hosp?, died within
    # 48h of out_time (likely in ICU or soon after), died within 90d after admission?
    demogi = demog['icustay_id'] == icustayid
    dem = np.array(
            list(demog.gender[demogi].values) +
            list(demog.age[demogi].values) +
            list(demog.elixhauser[demogi].values) +
            list(demog.re_admission[demogi].values) + 
            list(demog.morta_hosp[demogi].values) + 
            list(abs(demog.dod[demogi].values - demog.outtime[demogi].values) < (24*3600*2)) +
            list(demog.morta_90[demogi].values) + 
            [(qstime[icustayid][3] - qstime[icustayid][2])/3600]
            )
    
    #URINE OUTPUT
    iu = UO['icustay_id'] == icustayid   #rows of interest in inputMV
    output = UO[iu]    #subset of interest
    pread = UOpreadm[UOpreadm['icustay_id'] == icustayid-200000]['value'].values #preadmission UO ????????????????? Why no + 200000 for icustayid here?
    if len(pread) > 0:     #store the value, if available
        UOtot = np.nansum(pread)
    else:
        UOtot = 0
    #adding the volume of urine produced before start of recording!    
    UOnow = np.nansum(output[(output['charttime']>=t0) & (output['charttime'] <= t1)]['value'].values) #t0 and t1 defined above
    UOtot = np.nansum([UOtot, UOnow])
    
    for j in range(0, 48, timestep): #0:timestep: infinite
        t0 = 3600*j + beg   #left limit of time window
        t1 = 3600*(j + timestep) + beg   #right limit of time window
        ii = (temp[:, 2] >= t0) & (temp[:, 2] <= t1)  #index of items in this time period
        if sum(ii)>0:
            #ICUSTAY_ID, OUTCOMES, DEMOGRAPHICS
            reformat2[irow, 0] = (j/timestep)+1    # 'bloc' = timestep (1,2,3...)
            reformat2[irow, 1] = icustayid         # icustay_ID
            reformat2[irow, 2] = 3600*j+ beg       # t0 = lower limit of time window
            reformat2[irow, 3:11] = dem            # demographics and outcomes
            
            #CHARTEVENTS and LAB VALUES (+ includes empty cols for shock index and P/F)
            value = temp[ii]  # Records all values in this timestep
                
            if sum(ii) == 1:   # if only 1 row of values at this timestep
                reformat2[irow, 11:78] = value[:, 3:]
            else:
                reformat2[irow, 11:78] = np.nanmean(value[:, 3:], axis=0)  # mean of all available values
        
            # VASOPRESSORS
            # for CV: dose at timestamps.
            # for MV: 4 possibles cases, each one needing a different way to compute the dose of VP actually administered:
            #----t0---start----end-----t1----
            #----start---t0----end----t1----
            #-----t0---start---t1---end
            #----start---t0----t1---end----
            # MetaVision
            v = ((endv >= t0) & (endv <= t1)) | ((startv >= t0) & (endv<=t1)) | ((startv >= t0) & (startv <= t1))| ((startv <= t0) & (endv>=t1))
            # CareVue
            v2 = vaso2[(vaso2['charttime'] >= t0) & (vaso2['charttime'] <= t1)]['rate_std'].values
            if len(list(ratev[v]) +  list(v2)) > 0:
                v1 = np.nanmedian(list(ratev[v]) +  list(v2))
                v2 = np.nanmax(list(ratev[v]) + list(v2))
            else:
                v1 = np.nan
                v2 = np.nan
            
            if (~np.isnan(v1)) and (~np.isnan(v2)):
                reformat2[irow, 78] = v1    #median of dose of VP
                reformat2[irow, 79] = v2    #max dose of VP
                
                
            # heparin FLUID
            # Input from MV (4 ways to compute)
            infu = np.nansum(rate*(endt-startt)*((endt <= t1) & (startt >= t0))/3600 
                                          + rate*(endt-t0)*((startt <= t0) & (endt <= t1) & (endt >= t0))/3600 
                                          + rate*(t1-startt)*((startt >= t0) & (endt >= t1) & (startt <= t1))/3600 
                                          + rate*(t1-t0)*((endt >=t1) & (startt <= t0))/3600)
            # All boluses received during this timestep, from hepa_mv (need to check rate is NaN) and hepa_cv (simpler):
            bolus = np.nansum(input1[(np.isnan(input1[:, 5])) & (input1[:, 1] >= t0) & (input1[:, 1] <= t1), 6]) + np.nansum(input2[(input2[:, 1] >= t0) & (input2[:, 1] <= t1), 4])
            # Cumulate all fluid given
            totvol = np.nansum([totvol, infu, bolus])
            reformat2[irow, 80] = totvol    # Total fluid given, including pre-admission fluid; values accumulate recursively without resetting.
            reformat2[irow, 81] = np.nansum([infu, bolus])  #fluid given at this step
            
            # Urine Output
            UOnow = np.nansum(output[(output['charttime'] >= t0) & (output['charttime'] <= t1)]['value'].values)
            UOtot = np.nansum([UOtot, UOnow])
            reformat2[irow, 82] = UOtot    # Total Urine Output
            reformat2[irow, 83] = np.nansum(UOnow)  # Urine Output at this step
            
            irow+=1

reformat2 = np.delete(reformat2, range(irow, len(reformat2)) ,axis=0) 

In [ ]:
len(pd.Series(reformat2[:,1]).unique())

In [ ]:
np.save("./corpus/1hourly/reformat2", reformat2)

# 14. missing rate 체크

- 특히 aPTT에는 SAH를 적용안했으므로 결과가 매우 적나라하게 나올것임.
- 먼저 전체적인 missing rate를 체크하고 특히 aPTT값을 기준으로 데이터가 너무 없는 것들은 환자들은 제거시키자.
- 4h 간격으로 검사를 일반적으로 하니까 환자당 25% 정도의 aPTT 데이터는 있어야함. 근데 그러면 데이터가 너무 없어서 0.8을 thr로 했어..



- sepsis에서 하는 missing 체크는 SAH하고 하는거여서 어차피 의미 없음... 그냥 이전 논문 참고해서 state로 사용할 것들 구하자.

In [ ]:
reformat2 = np.load("./corpus/1hourly/reformat2.npy")

In [ ]:
reformat2 = reformat2[:,:84]

In [ ]:
########################################################################
#    CONVERT TO TABLE AND DELETE VARIABLES WITH EXCESSIVE MISSINGNESS
########################################################################
print('FULL ICU -- CONVERTING TO TABLE AND DELETE VARIABLES WITH EXCESSIVE MISSINGNESS')
dataheaders = ['Height_cm', 'Weight_kg', 'GCS','RASS','HR', 'SysBP', 'MeanBP', 'DiaBP',	'RR', 'SpO2', 'Temp_C', 'Temp_F', 'CVP', 'PAPsys', 'PAPmean', 'PAPdia', 'CI', 
'SVR', 'Interface', 'FiO2_100', 'FiO2_1', 'O2flow', 'PEEP', 'TidalVolume', 'MinuteVentil', 'PAWmean', 'PAWpeak', 'PAWplateau', 'Potassium', 'Sodium',
'Chloride', 'Glucose', 'BUN', 'Creatinine', 'Magnesium', 'Calcium', 'Ionised_Ca', 'CO2_mEqL', 'SGOT', 'SGPT', 'Total_bili', 'Direct_bili', 'Total_protein',
'Albumin', 'Troponin', 'CRP', 'Hb', 'Ht', 'RBC_count', 'WBC_count', 'Platelets_count', 'PTT', 'PT', 'ACT', 'INR', 'Arterial_pH', 'paO2', 'paCO2',
'Arterial_BE', 'Arterial_lactate', 'HCO3', 'ETCO2', 'SvO2', 'mechvent', 'extubated', 'Shock_Index', 'PaO2_FiO2']
dataheaders = ['bloc','icustayid','charttime','gender','age','elixhauser','re_admission', 'died_in_hosp', 'died_within_48h_of_out_time','mortality_90d','delay_end_of_record_and_discharge_or_death'] + \
    dataheaders + ['median_dose_vaso','max_dose_vaso','heparin_total','heparin_1hourly','output_total','output_1hourly']

reformat2t = pd.DataFrame(reformat2, columns=dataheaders)

In [ ]:
reformat2t.shape

In [ ]:
reformat2t['reward_PTT'] = reformat2t['PTT']

In [ ]:
miss = np.sum(np.isnan(reformat2), axis=0)/reformat2.shape[0]
for i,j,k in zip(range(85), dataheaders, miss):
    print(i,j,k)

In [ ]:
icu_id = reformat2t['icustayid'].drop_duplicates()
live_id = []
for i in icu_id:
    df = reformat2t.loc[reformat2t['icustayid']==i]
    ratio = np.sum(np.isnan(df['PTT']))/df.shape[0]
    if ratio <= 0.78:
        live_id.append(i)

print(len(live_id))
reformat2t = reformat2t.loc[reformat2t['icustayid'].isin(live_id)]  # Process only the IDs in live_id.

In [ ]:
reformat2t.shape

# 15. PTT값이 있는 마지막 index 까지만 자르기

- 기준 PTT값이 없는데 interpolation하면 정확하지 않음. 그래서 source가 있는 범위까지만 다룸.

In [ ]:
v = []
for i in live_id:
    df = reformat2t.loc[reformat2t['icustayid']==i]
    
    first_ptt_idx = df['PTT'].index[0]
    last_ptt_idx = df[~(df['PTT'].isna())]['PTT'].index[-1]
    save_n = last_ptt_idx-first_ptt_idx+1
    df_ = df.iloc[:save_n,:]
    v.append(np.array(df_))
    
reformat2 = np.vstack(v)
print(reformat2.shape)

In [ ]:
dataheaders.append("reward_PTT")
reformat3 = reformat2
reformat3t = pd.DataFrame(reformat2, columns=dataheaders)

# 16. Imputation

- 먼저 Linear imputation 수행.
    - 'mechvent'가 보이는 이유는 sepsis처리할 때 mechvent 전까지 범위로 imputation을 했기 때문임. (mechvent, exhausted..등은 missing 제거나 imputation에 넣지 않았기 때문.
    - 그런데 우리는 mechvent안쓰므로 모두 null값인 상태임. 남겨져있는 이유는 처리를 간편하게 하려고임. 마지막에 제거할 것임!
      
- 두 번째로 knn imputation 수행.

In [ ]:
# Do linear interpolation where missingness is low (kNN imputation doesnt work if all rows have missing values)
print('Full ICU -- Doing linear interpolation where missingness is low (kNN imputation doesnt work if all rows have missing values)')
miss = np.sum(np.isnan(reformat3), axis=0)/reformat3.shape[0]
ii = (miss>0) & (miss<0.05)  #less than 5% missingness
mechventcol = reformat3t.columns.tolist().index('mechvent')

for i in range(10,mechventcol): # Correct column by column
    if ii[i]==1:
        reformat3[:,i] = fixgaps(reformat3[:,i])

reformat3t[reformat3t.columns[10:mechventcol]] = reformat3[:,10:mechventcol]

# KNN IMPUTATION -  Done on chunks of 10K records.
print('Full ICU -- KNN imputation')

reformat3t_cols = reformat3t.columns.tolist()
mechventcol = reformat3t_cols.index('mechvent')
ref = np.copy(reformat3[:,11:mechventcol])  #columns of interest

bar_knn = pyprind.ProgBar(len(range(0,reformat3.shape[0],9999)))
for i in range(0,reformat3.shape[0],9999):   #dataset divided in 10K rows chunks (otherwise too large)
    bar_knn.update()
    ref[i:i+9999,:] = KNN(k=1).fit_transform(ref[i:i+9999,:])

reformat3t[reformat3t_cols[11:mechventcol]] = ref 

reformat4t = reformat3t.copy()
reformat4 = reformat4t.values


In [ ]:
reformat4.shape

In [ ]:
len(reformat4t['icustayid'].unique())

In [ ]:
len(pd.Series(reformat4[:,1]).unique())

# 17. 다른 변수로부터 파생된 변수들 계산

- P/F, Shock Index, SOFA.. 등등은 다른 변수들을 이용해서 구해야함.
    - ex. PaO2_FiO2 = PaO2 / FiO2

In [ ]:
print('FULL ICU -- COMPUTING SOME DERIVED VARIABLES: P/F, Shock Index, SOFA, SIRS...')
# CORRECT GENDER
reformat4t['gender'] = reformat4t['gender'] - 1

# CORRECT AGE > 200 yo
ii = reformat4t['age'] > 150*365.25
reformat4t.loc[ii,'age'] = 91.4*365.25


# FIX Elixhauser missing values
reformat4t['elixhauser'].loc[np.isnan(reformat4t['elixhauser'])] = np.nanmedian(reformat4t['elixhauser'])   #use the median value / only a few missing data points 


# Recompute P/F with no missing values...
reformat4t['PaO2_FiO2'] = reformat4t['paO2']/reformat4t['FiO2_1']

# Recompute SHOCK INDEX without NAN and INF
reformat4t['Shock_Index'] = reformat4t['HR']/reformat4t['SysBP']

reformat4t.loc[np.isinf(reformat4t['Shock_Index']), 'Shock_Index'] = np.NaN

d = np.nanmean(reformat4t['Shock_Index'])
reformat4t['Shock_Index'].fillna(d, inplace=True)

# SOFA - at each timepoint we need (in this order):  
# P/F,  MV,  PLT,  TOT_BILI,  MAP,  NORAD(max),  GCS,  CR,  UO
s = reformat4t[['PaO2_FiO2', 'Platelets_count', 'Total_bili', 'MeanBP', 'max_dose_vaso', 'GCS', 'Creatinine', 'output_1hourly']].values
p = np.arange(5)
s1=np.array([s[:,0]>400, (s[:, 0]>=300) & (s[:, 0]<400), (s[:, 0]>=200) & (s[:, 0]<300), (s[:, 0]>=100) & (s[:, 0]<200), s[:, 0]<100 ])   #count of points for all 6 criteria of sofa
s2=np.array([s[:,1]>150, (s[:, 1]>=100) & (s[:, 1]<150), (s[:, 1]>=50) & (s[:, 1]<100), (s[:, 1]>=20) & (s[:, 1]<50), s[:, 1]<20 ])
s3=np.array([s[:, 2]<1.2, (s[:, 2]>=1.2) & (s[:, 2]<2), (s[:, 2]>=2) & (s[:, 2]<6), (s[:, 2]>=6) & (s[:, 2]<12), s[:, 2]>12 ])
s4=np.array([s[:, 3]>=70, (s[:, 3]<70) & (s[:, 3]>=65), (s[:, 3]<65), (s[:, 4]>0) & (s[:, 4]<=0.1), s[:, 4]>0.1 ])
s5=np.array([s[:, 5]>14, (s[:, 5]>12) & (s[:, 5]<=14), (s[:, 5]>9) & (s[:, 5]<=12), (s[:, 5]>5) & (s[:, 5]<=9), s[:, 5]<=5])
s6=np.array([s[:, 6]<1.2, (s[:, 6]>=1.2) & (s[:, 6]<2), (s[:, 6]>=2) & (s[:, 6]<3.5), ((s[:, 6]>=3.5) & (s[:, 6]<5))|(s[:, 7]<84), (s[:, 6]>5)|(s[:, 7]<34)])

num_columns = reformat4t.shape[1]   #nr of variables in data
newcols_reformat4 = np.zeros((reformat4t.shape[0],7)) 
for i in range(reformat4t.shape[0]):
    t = max(p[s1[:, i]], default=0) + max(p[s2[:, i]], default=0) + max(p[s3[:, i]], default=0) + max(p[s4[:, i]], default=0) + max(p[s5[:, i]], default=0) + max(p[s6[:, i]], default=0)  #SUM OF ALL 6 CRITERIA
    if t > 0:
        newcols_reformat4[i, :] = [max(p[s1[:, i]], default=0), max(p[s2[:, i]], default=0), max(p[s3[:, i]], default=0), max(p[s4[:, i]], default=0), max(p[s5[:, i]], default=0), max(p[s6[:, i]], default=0), t]

# SIRS - at each timepoint |  need: temp HR RR PaCO2 WBC 
s = reformat4t[['Temp_C', 'HR', 'RR', 'paCO2', 'WBC_count']].values

s1=np.array([(s[:, 0]>=38) | (s[:,0]<=36)])   # Count of points for all criteria of SIRS
s2=np.array([s[:, 1]>90])
s3=np.array([(s[:, 2]>=20) | (s[:, 3]<=32)])
s4=np.array([(s[:, 4]>=12) | (s[:, 4]<4)])
newcols_sirs = (1*s1) + (1*s2) + (1*s3) + (1*s4)

# Adds 2 cols for SOFA and SIRS
# Records values
reformat4t['SOFA'] = newcols_reformat4[:,-1]
reformat4t['SIRS'] = newcols_sirs[0]

# 18. 과도한 결과값을 가지는 환자들은 제거.

- outlier 값 가지고 있는 환자는 그냥 제거해버림!! MDP사이에 구멍이 생겨서..
- 근데 좀 그냥 너무 제거한거 같아서 10000으로 함. ddpg 보면 max가 1000 이여서 push로 10000정도는 들어갈 수 있는것으로 보임.

In [ ]:
a = reformat4t['heparin_1hourly'].values
plt.boxplot(a)

In [ ]:
Q3 = np.quantile(a, 0.75)
Q1 = np.quantile(a, 0.25)

# thr = Q3 + 1.5 *(Q3-Q1)

# The previous filter may remove too many rows. The referenced DDPG paper uses a maximum action of 1000 units/ml; use 10000 here.
thr = 10000

In [ ]:
# reformat4t_copy = reformat4t.copy()
# reformat4t = reformat4t_copy

In [ ]:
reformat4t = reformat4t.loc[reformat4t['heparin_1hourly']<=thr,:]

In [ ]:
len(reformat4t['icustayid'].unique())

In [ ]:
########################################################################
#                   EXCLUSION OF SOME PATIENTS 
########################################################################
a = reformat4t['heparin_1hourly'] > thr 
i = reformat4t[a]['icustayid'].unique() 
i = reformat4t['icustayid'].isin(i) 
reformat4t.drop(reformat4t.index[i], inplace=True) 

# Check for patients with extreme UO = outliers = to be deleted (>40 litres of UO per 4h!!)
a = reformat4t['output_1hourly'] > 9000 
i = reformat4t[a]['icustayid'].unique() 
i = reformat4t['icustayid'].isin(i) 
reformat4t.drop(reformat4t.index[i], inplace=True) 

# Some have bili = 999999
a = reformat4t['Total_bili'] > 10000
i = reformat4t[a]['icustayid'].unique()
i = reformat4t['icustayid'].isin(i)
reformat4t.drop(reformat4t.index[i], inplace=True)

# Check for patients with extreme INTAKE = outliers = to be deleted 
a = reformat4t['heparin_1hourly'] > 25000 
i = reformat4t[a]['icustayid'].unique()
i = reformat4t['icustayid'].isin(i) 
reformat4t.drop(reformat4t.index[i], inplace=True)



In [ ]:
print(reformat4t.shape)

In [ ]:
len(reformat4t['icustayid'].drop_duplicates())

In [ ]:
reformat4t

# 19. heparin cohort 생성

- icustayid', 'mean_PTT', 'mean_3hourly_heparin', 'max_sofa', 'max_sirs', 'first_heparin_time'

In [ ]:
#######################################################################
#       CREATE HEPARIN COHORT FROM ALL ICU PATIENTS EXTRACTED
########################################################################
print('Creating heparin cohort')
# Create array with 1 row per icu admission
# Keep only patients with flagged heparin
# Assumed baseline SOFA is zero
heparin = np.zeros((30000,6)) #NOTE: For other cohorts, this size may have to be changed
irow = 0

for icustayid in tqdm(icustayidlist):
    ii = reformat4t['icustayid'] == icustayid 
    if sum(ii) > 0:
        sofa = reformat4t['SOFA'][ii]
        sirs = reformat4t['SIRS'][ii]
        heparin[irow, 0] = icustayid
        heparin[irow, 1] = np.mean(reformat4t['PTT'][ii]) # mean PTT
        heparin[irow, 2] = np.mean(reformat4t['heparin_1hourly'][ii]) # mean heparin 3 hourly
        heparin[irow, 3] = np.max(sofa)
        heparin[irow, 4] = np.max(sirs)
        heparin[irow, 5] = qstime[icustayid][0]   #time of onset of heparin time
        irow+=1

heparin = np.delete(heparin, range(irow, len(heparin)) ,axis=0) # Remove extra rows
heparin = pd.DataFrame(heparin, columns=['icustayid', 'mean_PTT', 'mean_1hourly_heparin', 'max_sofa', 'max_sirs', 'first_heparin_time'])

# Final count of patients included
print('Final patient count:', heparin.shape[0])  

In [ ]:
reformat4t.to_csv("./corpus/1hourly/reformat4t.csv")

In [ ]:
heparin.to_csv('./final_result/1hourly_discrete/new_heparin_mimiciii.csv', index=False) 

# 19. action, state 등을 정의하는데 필요한 df 생성

    - 여기서부터는 10~18 과정과 거의 동일하며, 만들어진 heparin cohort를 기준으로 action이나 state에 필요한 정보만 따로 모아서 만들기 위해 수행하는 과정임.
    - 동일한 부분은 요약 생략하고 다른 부분만 요약함.

In [ ]:
########################################################################
#           INITIAL REFORMAT WITH CHARTEVENTS, LABS AND MECHVENT
########################################################################
#gives an array with all unique charttime (1 per row) and all items in columns.
################## IMPORTANT !!!!!!!!!!!!!!!!!!
# Here we use -24 -> +48 to define the MDP
print('Heparin Cohort -- Making an array with all unique charttime (1 per row) and all items in columns.')
reformat = np.nan*np.ones((2000000,69))  #final table 
qstime = dict()
winb4 = 0   #lower limit for inclusion of data (24h before time flag)
winaft = 48  # upper limit (96h after)
irow = 0  #recording row for summary table

for icustayidrow in tqdm(range(1, heparin.shape[0]+1)):
    qst = heparin['first_heparin_time'].iloc[icustayidrow-1] #;%,3); %flag for presumed infection
    icustayid = int(heparin['icustayid'].iloc[icustayidrow-1])
    qstime[icustayid] = np.zeros(4)
    # CHARTEVENTS
    if (icustayid-200000) < 10000:
        temp=ce010
    elif (icustayid-200000) < 20000:
        temp=ce1020
    elif (icustayid-200000) < 30000:
        temp=ce2030
    elif (icustayid-200000) < 40000:
        temp=ce3040
    elif (icustayid-200000) < 50000:
        temp=ce4050
    elif (icustayid-200000) < 60000:
        temp=ce5060
    elif (icustayid-200000) < 70000:
        temp=ce6070
    elif (icustayid-200000) < 80000:
        temp=ce7080
    elif (icustayid-200000) < 90000:
        temp=ce8090
    else:
        temp=ce90100
    temp = temp[temp['icustay_id'] == icustayid]

    ii = (temp['charttime'] >= qst)& (temp['charttime'] <= qst + (winaft)*3600)
    temp = temp.loc[ii]   #only time period of interest

    # LAB EVENTS
    ii = labU['icustay_id'] == icustayid
    temp2 = labU.loc[ii]
    ii = (temp2['charttime'] >= qst) & (temp2['charttime'] <= qst + (winaft)*3600)
    temp2 = temp2.loc[ii]   #only time period of interest

    
    t = np.unique(pd.concat([temp['charttime'], temp2['charttime']], ignore_index=True).values) # List of unique timestamps from all 3 sources / sorted in ascending order

    if len(t) > 0:
        for i in range(len(t)):
            # CHARTEVENTS
            ii = temp['charttime'] == t[i]
            col = temp.loc[ii, 'itemid']
            value = temp.loc[ii, 'valuenum']
            reformat[irow, 0] = i+1 # Timestep  
            reformat[irow, 1] = icustayid
            reformat[irow, 2] = t[i] # Charttime
            reformat[irow, 3] = qst  # Store the presumed onset time
            reformat[irow, 3+col.astype(int).values] = value.values # Store available values

            #LAB VALUES
            ii = temp2['charttime'] == t[i]
            col = temp2.loc[ii, 'itemid']
            value = temp2.loc[ii, 'valuenum']
            reformat[irow,31+col.astype(int).values] = value.values  #store available values
        
            irow += 1

        qstime[icustayid][0] = qst # Flag for presumed infusion
        # WE SAVE FIRST and LAST TIMESTAMPS, in QSTIME, for each ICUSTAYID
        qstime[icustayid][1] = t[0]   # First timestamp
        qstime[icustayid][2] = t[-1]  # Last timestamp
        qstime[icustayid][3] = demog.loc[demog['icustay_id'] == icustayid, 'dischtime'].values[0] # Discharge time

reformat = np.delete(reformat, range(irow, len(reformat)) ,axis=0)  # Remove unused rows


In [ ]:
########################################################################
#                                   OUTLIERS 
########################################################################
print('Heparin Cohort -- Handling outliers')

# Weight
reformat = deloutabove(reformat, 5, 300)

# Heart Rate
reformat = deloutabove(reformat, 8, 250)

# Blood Pressure
reformat = deloutabove(reformat, 9, 300)
reformat = deloutbelow(reformat, 10, 0)
reformat = deloutabove(reformat, 11, 200)
reformat = deloutbelow(reformat, 11, 0)
reformat = deloutabove(reformat, 11, 200)

# Respiratory Rate
reformat = deloutabove(reformat, 12, 80) 

# SpO2
reformat = deloutabove(reformat, 13, 150)
reformat[reformat[:, 13]>100, 13] = 100 
reformat = deloutbelow(reformat, 13, 50) 

# Temperature
reformat[(reformat[:, 14] > 90) & (np.isnan(reformat[:, 15])), 15] = reformat[(reformat[:, 14] > 90) & (np.isnan(reformat[:, 15])), 14]
reformat = deloutabove(reformat, 14, 90) 
reformat = deloutbelow(reformat, 14, 25) 

# interface / is in col 23
# FiO2
reformat = deloutabove(reformat, 23, 100)
reformat[reformat[:, 23] < 1 , 23] = reformat[reformat[:, 23] < 1 , 23]*100
reformat = deloutbelow(reformat, 23, 20) 
reformat = deloutabove(reformat, 24, 1.5)

# O2 FLOW
reformat = deloutabove(reformat, 25, 70)

# PEEP
reformat=deloutbelow(reformat, 26, 0)
reformat=deloutabove(reformat, 26, 40)

# Total Volume
reformat=deloutabove(reformat, 27, 1800)

# Mean Volume
reformat=deloutabove(reformat, 28, 50)

# Potassium
reformat=deloutbelow(reformat, 32, 1)
reformat=deloutabove(reformat, 32, 15)

# Sodium
reformat=deloutbelow(reformat, 33, 95)
reformat=deloutabove(reformat, 33, 178)

# Chloride
reformat=deloutbelow(reformat, 34, 70)
reformat=deloutabove(reformat, 34, 150)

# Glucose
reformat=deloutbelow(reformat, 35, 1)
reformat=deloutabove(reformat, 35, 1000) 

# Creatinine
reformat=deloutabove(reformat, 37, 150)

# Magnesium
reformat=deloutabove(reformat, 38, 10)

# Calcium
reformat=deloutabove(reformat, 39, 20)

# Ionized Calcium
reformat=deloutabove(reformat, 40, 5)

# CO2
reformat=deloutabove(reformat, 41, 120)

# SGPT/SGOT
reformat=deloutabove(reformat, 42, 10000)
reformat=deloutabove(reformat, 43, 10000)

# Hb/Ht
reformat=deloutabove(reformat, 50, 20)
reformat=deloutabove(reformat, 51, 65)

# White Blood Cells
reformat=deloutabove(reformat, 53, 500)

# Platelets
reformat=deloutabove(reformat, 54, 2000)

# INR
reformat=deloutabove(reformat, 58, 20)

# pH
reformat=deloutbelow(reformat, 59, 6.7)
reformat=deloutabove(reformat, 59, 8) 

# pO2
reformat=deloutabove(reformat, 60, 700)

# pCO2
reformat=deloutabove(reformat, 61, 200)

# Base Excess
reformat=deloutbelow(reformat, 62, -50)

# Lactate
reformat=deloutabove(reformat, 63, 30)

####################################################################
# More data manipulation / imputation from existing values
# Estimate GCS from RASS - data from Wesley JAMA 2003
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] >= 0), 6] = 15
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -1), 6] = 14
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -2), 6] = 12
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -3), 6] = 11
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -4), 6] = 6
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -5), 6] = 3

# FiO2
reformat[(~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 24])), 24] = reformat[(~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 24])), 23] / 100
reformat[(~np.isnan(reformat[:, 24])) & (np.isnan(reformat[:, 23])), 23] = reformat[(~np.isnan(reformat[:, 24])) & (np.isnan(reformat[:, 23])), 24] * 100

# ESTIMATE FiO2 /// with use of interface / device (cannula, mask, ventilator....)
print('Heparin Cohort -- Doing sample and hold')
reformatsah = SAH(reformat, sample_and_hold, adjust=1) # do SAH first to handle this task, setting `adjust` to 1 since we added another column

# NO FiO2, YES O2 flow, no interface OR cannula
ii = np.where((np.isnan(reformatsah[:,23])) & (~np.isnan(reformatsah[:,25])) & ((reformatsah[:,22] == 0) | (reformatsah[:,22] == 2)))[0] #As np.where returns a tuple
reformat[ii[reformatsah[ii, 25] <= 15], 23] = 70
reformat[ii[reformatsah[ii, 25] <= 12], 23] = 62
reformat[ii[reformatsah[ii, 25] <= 10], 23] = 55
reformat[ii[reformatsah[ii, 25] <= 8], 23]  = 50
reformat[ii[reformatsah[ii, 25] <= 6], 23]  = 44
reformat[ii[reformatsah[ii, 25] <= 5], 23]  = 40
reformat[ii[reformatsah[ii, 25] <= 4], 23]  = 36
reformat[ii[reformatsah[ii, 25] <= 3], 23]  = 32
reformat[ii[reformatsah[ii, 25] <= 2], 23]  = 28
reformat[ii[reformatsah[ii, 25] <= 1], 23]  = 24

# NO FiO2, NO O2 flow, no interface OR cannula
ii = np.where((np.isnan(reformatsah[:, 23])) & np.isnan(reformatsah[:, 25]) & ((reformatsah[:, 22] == 0) | (reformatsah[:, 22] == 2)))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 23] = 21

# NO FiO2, YES O2 flow, face mask OR.... OR ventilator (assume it's face mask)
ii = np.where((np.isnan(reformatsah[:,23])) & (~np.isnan(reformatsah[:,25])) & 
((reformatsah[:, 22]==1) | (reformatsah[:, 22]==3) | (reformatsah[:, 22]==4) | (reformatsah[:, 22]==5) | (reformatsah[:, 22]==6) | (reformatsah[:, 22]==9) | (reformatsah[:, 22]==10)))[0]
reformat[ii[reformatsah[ii, 25]<=15], 23] = 75
reformat[ii[reformatsah[ii, 25]<=12], 23] = 69
reformat[ii[reformatsah[ii, 25]<=10], 23] = 66
reformat[ii[reformatsah[ii, 25]<=8], 23]  = 58
reformat[ii[reformatsah[ii, 25]<=6], 23]  = 40
reformat[ii[reformatsah[ii, 25]<=4], 23]  = 36

# NO FiO2, NO O2 flow, face mask OR ....OR ventilator
ii = np.where(np.isnan(reformatsah[:, 23]) & np.isnan(reformatsah[:, 25]) & ((reformatsah[:, 22] == 1) | (reformatsah[:, 22] == 3) | 
(reformatsah[:, 22] == 4) | (reformatsah[:, 22] == 5) | (reformatsah[:, 22] == 6) | (reformatsah[:, 22] == 9) | (reformatsah[:, 22] == 10)))[0]  # No FiO2 given and O2 flow given, no interface OR cannula
reformat[ii, 23] = np.nan

# NO FiO2, YES O2 flow, Non rebreather mask
ii = np.where(np.isnan(reformatsah[:, 23]) & (~np.isnan(reformatsah[:, 25])) & (reformatsah[:, 22] == 7))[0]
reformat[ii[reformatsah[ii, 25] >= 10], 23] = 90
reformat[ii[reformatsah[ii, 25] >= 15], 23] = 100
reformat[ii[reformatsah[ii, 25] < 10], 23]  = 80
reformat[ii[reformatsah[ii, 25] <= 8], 23]  = 70
reformat[ii[reformatsah[ii, 25] <= 6], 23]  = 60

# NO FiO2, NO O2 flow, NRM
ii= np.where(np.isnan(reformatsah[:, 23]) & np.isnan(reformatsah[:, 25]) & (reformatsah[:, 22]==7))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 23] = np.nan

# Update Fi02 columns again
ii = (~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 24]))
reformat[ii, 24] = reformat[ii, 23]/100
ii = (~np.isnan(reformat[:, 24])) & (np.isnan(reformat[:, 24]))
reformat[ii, 23] = reformat[ii, 24]*100

# BLOOD PRESSURE
ii = (~np.isnan(reformat[:, 9])) & (~np.isnan(reformat[:, 10])) & np.isnan(reformat[:, 11])
reformat[ii, 11] = (3*reformat[ii, 10] - reformat[ii, 9])/2
ii = (~np.isnan(reformat[:, 9])) & (~np.isnan(reformat[:, 11])) & np.isnan(reformat[:, 10])
reformat[ii, 10] = (reformat[ii, 9] + 2*reformat[ii, 11])/3
ii = (~np.isnan(reformat[:, 10])) & (~np.isnan(reformat[:, 11])) & np.isnan(reformat[:, 9])
reformat[ii, 9] = 3*reformat[ii, 10] - 2*reformat[ii, 11]

# TEMPERATURE
# some values recorded in the wrong column
ii = (reformat[:, 15] > 25) & (reformat[:, 15] < 45) #tempF close to 37deg??!
reformat[ii, 14] = reformat[ii, 15]
reformat[ii, 15] = np.nan
ii = reformat[:, 14] >70  # TempC > 70, likely recorded in Farenheit
reformat[ii, 15] = reformat[ii, 14]
reformat[ii, 14] = np.nan
ii = (~np.isnan(reformat[:, 14])) & np.isnan(reformat[:, 15])
reformat[ii, 15] = reformat[ii, 14] * 1.8 + 32
ii = (~np.isnan(reformat[:, 14])) & np.isnan(reformat[:, 13])
reformat[ii, 14] = (reformat[ii, 15] - 32)/1.8

# Hb/Ht
ii = (~np.isnan(reformat[:, 50])) & np.isnan(reformat[:,51])
reformat[ii, 51] = (reformat[ii, 50] * 2.862) + 1.216
ii = (~np.isnan(reformat[:, 51])) & np.isnan(reformat[:,50])
reformat[ii, 50] = (reformat[ii, 51] - 1.216)/2.862

# BILIRUBIN
ii = (~np.isnan(reformat[:, 44])) & np.isnan(reformat[:,45])
reformat[ii, 45] = (reformat[ii, 44]*0.6934)-0.1752
ii = (~np.isnan(reformat[:, 45])) & np.isnan(reformat[:,44])
reformat[ii, 44] = (reformat[ii, 45] + 0.1752)/0.6934

In [ ]:
########################################################################
#                      SAMPLE AND HOLD on RAW DATA
########################################################################
print('heparin COHORT -- SAMPLE AND HOLD on RAW DATA')
reformat = SAH(reformat[:, 0:69], sample_and_hold, adjust=1)  # Setting `adjust` to 1 to account for the added column for `presumed_onset`
#######################################################################

In [ ]:
print('Full ICU -- Data combination')
# WARNING: the time window of interest has been defined above (here 0 -> 96)! 
timestep = 1  # Resolution of timesteps, in hours
irow = 0
icustayidlist = np.unique(reformat[:,1]).astype(np.int32)
reformat2 = np.nan*np.ones((reformat.shape[0], 86))  # Output array  

num_patients = len(icustayidlist)  # Number of patients
# Adding 2 empty cols for future shock index=HR/SBP and P/F
reformat = np.hstack([reformat, np.nan*np.ones((reformat.shape[0], 2))])

for i in tqdm(range(len(icustayidlist))):
    icustayid = icustayidlist[i] 
    
    #CHARTEVENTS AND LAB VALUES
    temp = reformat[reformat[:,1] == icustayid,:]   # subtable of interest
    beg = temp[0,2]   # timestamp of first record
    
    #heparin IV FLUID STUFF
    iv = hepa_mv['icustay_id'] == icustayid         # rows of interest in heparin_mv
    input = hepa_mv[iv]                             # subset of interest
    iv = hepa_cv['icustay_id'] == icustayid         # rows of interest in heparin_cv
    input2 = hepa_cv[iv]                            # subset of interest
    startt = input['starttime']                     # start of all infusions and boluses
    endt = input['endtime']                         # end of all infusions and boluses
    rate = input['norm_rate_of_infusion']    
    
    input = input.values
    input2 = input2.values
    
    
    # Compute volume of fluid given before start of record!!!
    t0 = 0
    t1 = beg
    # Input from MetaVision (4 ways to compute)
    infu = np.nansum(rate*(endt-startt)*((endt<=t1) & (startt >= t0))/3600 + rate*(endt-t0)*((startt <= t0) & (endt <= t1) & (endt >= t0))/3600 +
                            rate*(t1-startt)*((startt >= t0) & (endt >=t1) & (startt <= t1))/3600 + rate*(t1-t0)*((endt >= t1) & (startt <= t0))/3600)
    # All boluses received during this timestep, from heparin_mv (need to check rate is NaN) and heparin_cv (simpler):
    bolus = np.nansum(input[np.isnan(input[:, 5]) & (input[:, 1] >= t0) & (input[:, 1] <= t1), 6]) + np.nansum(input2[(input2[:, 1] >= t0) & (input2[:, 1] <= t1), 4])
    totvol = np.nansum([infu,bolus])
    
    
    #########################################################################################
    #VASOPRESSORS    
    iv = vasoMV['icustay_id'] == icustayid  # rows of interest in vasoMV
    vaso1 = vasoMV[iv]    # subset of interest
    iv = vasoCV['icustay_id'] == icustayid   # rows of interest in vasoCV
    vaso2 = vasoCV[iv]    # subset of interest
    startv = vaso1['starttime'].values  # start of VP infusion
    endv = vaso1['endtime'].values      # end of VP infusions
    ratev = vaso1['rate_std'].values  # rate of VP infusion
    
    # DEMOGRAPHICS / gender, age, elixhauser, re-admit, died in hosp?, died within
    # 48h of out_time (likely in ICU or soon after), died within 90d after admission?
    demogi = demog['icustay_id'] == icustayid
    dem = np.array(
            list(demog.gender[demogi].values) +
            list(demog.age[demogi].values) +
            list(demog.elixhauser[demogi].values) +
            list(demog.re_admission[demogi].values) + 
            [temp[0,3]] +
            list(demog.morta_hosp[demogi].values) + 
            list(abs(demog.dod[demogi].values - demog.outtime[demogi].values) < (24*3600*2)) +
            list(demog.morta_90[demogi].values) + 
            [(qstime[icustayid][3] - qstime[icustayid][2])/3600]
            )
    
    #URINE OUTPUT
    iu = UO['icustay_id'] == icustayid   #rows of interest in heparin_mv
    output = UO[iu]    #subset of interest
    pread = UOpreadm[UOpreadm['icustay_id'] == icustayid-200000]['value'].values #preadmission UO ????????????????? Why no + 200000 for icustayid here?
    if len(pread) > 0:     #store the value, if available
        UOtot = np.nansum(pread)
    else:
        UOtot = 0
    #adding the volume of urine produced before start of recording!    
    UOnow = np.nansum(output[(output['charttime']>=t0) & (output['charttime'] <= t1)]['value'].values) #t0 and t1 defined above
    UOtot = np.nansum([UOtot, UOnow])
    
    for j in range(0, 48, timestep): #0:timestep:100 #0 until 99 = 100 hours in total
        t0 = 3600*j + beg   #left limit of time window
        t1 = 3600*(j + timestep) + beg   #right limit of time window
        ii = (temp[:, 2] >= t0) & (temp[:, 2] <= t1)  #index of items in this time period
        if sum(ii)>0:
            #ICUSTAY_ID, OUTCOMES, DEMOGRAPHICS
            reformat2[irow, 0] = (j/timestep)+1    # 'bloc' = timestep (1,2,3...)
            reformat2[irow, 1] = icustayid         # icustay_ID
            reformat2[irow, 2] = 3600*j+ beg       # t0 = lower limit of time window
            reformat2[irow, 3:12] = dem            # demographics and outcomes
            
            #CHARTEVENTS and LAB VALUES (+ includes empty cols for shock index and P/F)
            value = temp[ii]  # Records all values in this timestep
                
            if sum(ii) == 1:   # if only 1 row of values at this timestep
                reformat2[irow, 12:79] = value[:, 4:]
            else:
                reformat2[irow, 12:79] = np.nanmean(value[:, 4:], axis=0)  # mean of all available values
        
            # VASOPRESSORS
            # for CV: dose at timestamps.
            # for MV: 4 possibles cases, each one needing a different way to compute the dose of VP actually administered:
            #----t0---start----end-----t1----
            #----start---t0----end----t1----
            #-----t0---start---t1---end
            #----start---t0----t1---end----
            # MetaVision
            v = ((endv >= t0) & (endv <= t1)) | ((startv >= t0) & (endv<=t1)) | ((startv >= t0) & (startv <= t1))| ((startv <= t0) & (endv>=t1))
            # CareVue
            v2 = vaso2[(vaso2['charttime'] >= t0) & (vaso2['charttime'] <= t1)]['rate_std'].values
            if len(list(ratev[v]) +  list(v2)) > 0:
                v1 = np.nanmedian(list(ratev[v]) +  list(v2))
                v2 = np.nanmax(list(ratev[v]) + list(v2))
            else:
                v1 = np.nan
                v2 = np.nan
            
            if (~np.isnan(v1)) and (~np.isnan(v2)):
                reformat2[irow, 79] = v1    #median of dose of VP
                reformat2[irow, 80] = v2    #max dose of VP
                
                
            # INPUT FLUID
            # Input from MV (4 ways to compute)
            infu = np.nansum(rate*(endt-startt)*((endt <= t1) & (startt >= t0))/3600 
                                          + rate*(endt-t0)*((startt <= t0) & (endt <= t1) & (endt >= t0))/3600 
                                          + rate*(t1-startt)*((startt >= t0) & (endt >= t1) & (startt <= t1))/3600 
                                          + rate*(t1-t0)*((endt >=t1) & (startt <= t0))/3600)
            # All boluses received during this timestep, from heparin_mv (need to check rate is NaN) and heparin_cv (simpler):
            bolus = np.nansum(input[(np.isnan(input[:, 5])) & (input[:, 1] >= t0) & (input[:, 1] <= t1), 6]) + np.nansum(input2[(input2[:, 1] >= t0) & (input2[:, 1] <= t1), 4])
            # Cumulate all fluid given
            totvol = np.nansum([totvol, infu, bolus])
            reformat2[irow, 81] = totvol    # Total fluid given, including pre-admission fluid; values accumulate recursively without resetting.
            reformat2[irow, 82] = np.nansum([infu, bolus])  #fluid given at this step
            
            # Urine Output
            UOnow = np.nansum(output[(output['charttime'] >= t0) & (output['charttime'] <= t1)]['value'].values)
            UOtot = np.nansum([UOtot, UOnow])
            reformat2[irow, 83] = UOtot    # Total Urine Output
            reformat2[irow, 84] = np.nansum(UOnow)  # Urine Output at this step
            
            irow+=1

reformat2 = np.delete(reformat2, range(irow, len(reformat2)) ,axis=0) 

In [ ]:
reformat2 = reformat2[:,:85]

In [ ]:
#########################################################################
#             CONVERT TO TABLE AND KEEP ONLY WANTED VARIABLE
#########################################################################

dataheaders = [i[1:-1] for i in sample_and_hold.columns]
dataheaders.extend(['Shock_Index', 'PaO2_FiO2'])
dataheaders = ['bloc', 'icustayid', 'charttime', 'gender', 'age', 'elixhauser', 're_admission', 'presumed_onset', 'died_in_hosp', \
    'died_within_48h_of_out_time', 'mortality_90d', 'delay_end_of_record_and_discharge_or_death'] + \
    dataheaders + ['median_dose_vaso','max_dose_vaso','heparin_total','heparin_1hourly','output_total','output_1hourly']


reformat2t = pd.DataFrame(reformat2, columns=dataheaders)

# headers I want to keep
dataheaders5 = ['bloc', 'icustayid', 'charttime', 'gender', 'age', 'elixhauser', 're_admission', 'presumed_onset', 'died_in_hosp', 'died_within_48h_of_out_time', \
    'mortality_90d', 'delay_end_of_record_and_discharge_or_death', 'Weight_kg', 'GCS', 'HR', 'SysBP', 'MeanBP', 'DiaBP', 'RR', 'SpO2', 'Temp_C', 'FiO2_1', \
    'Potassium', 'Sodium', 'Chloride', 'Glucose', 'BUN', 'Creatinine', 'Magnesium', 'Calcium', 'Ionised_Ca', 'CO2_mEqL', 'SGOT', 'SGPT', 'Total_bili', 'Albumin', \
    'Hb', 'WBC_count', 'Platelets_count', 'PTT', 'PT', 'INR', 'Arterial_pH', 'paO2', 'paCO2', 'Arterial_BE', 'HCO3', 'Arterial_lactate', 'mechvent', 'Shock_Index', \
    'PaO2_FiO2', 'median_dose_vaso', 'max_dose_vaso', 'heparin_total', 'heparin_1hourly', 'output_total', 'output_1hourly']

reformat3t = reformat2t[dataheaders5].copy() 

## SOME DATA MANIPULATION BEFORE IMPUTATION
#CORRECT GENDER
reformat3t['gender'] = reformat3t['gender'] - 1

# CORRECT AGE > 200 yo
ii = reformat3t['age'] > 150*365.25
reformat3t.loc[ii,'age'] = 91.4*365.25

# FIX Elixhauser missing values
reformat3t['elixhauser'].loc[np.isnan(reformat3t['elixhauser'])] = np.nanmedian(reformat3t['elixhauser'])   #use the median value / only a few missing data points 

# Vasopressors / no NAN
reformat3t['median_dose_vaso'].fillna(0, inplace=True)
reformat3t['max_dose_vaso'].fillna(0, inplace=True)

In [ ]:
reformat3t['reward_PTT'] = reformat3t['PTT']

In [ ]:
live_id = reformat3t['icustayid'].drop_duplicates()

In [ ]:
print(len(live_id))

In [ ]:
v = []
for i in live_id:
    df = reformat3t.loc[reformat3t['icustayid']==i]
    
    first_ptt_idx = df['PTT'].index[0]
    last_ptt_idx = df[~(df['PTT'].isna())]['PTT'].index[-1]
    save_n = last_ptt_idx-first_ptt_idx+1
    df_ = df.iloc[:save_n,:]
    v.append(np.array(df_))
    
reformat2_ = np.vstack(v)
print(reformat2_.shape)

In [ ]:
dataheaders5.append("reward_PTT")
reformat3 = reformat2_
reformat3t = pd.DataFrame(reformat3, columns=dataheaders5)

In [ ]:
reformat3t['Shock_Index'] = np.zeros(reformat3t.shape[0])
reformat3t['PaO2_FiO2'] = np.zeros(reformat3t.shape[0])

In [ ]:
########################################################################
#        HANDLING OF MISSING VALUES & CREATE REFORMAT4T
########################################################################

# Do linear interpolation where missingness is low (kNN imputation doesnt work if all rows have missing values)
reformat3 = reformat3t.values
miss = np.sum(np.isnan(reformat3), axis=0)/reformat3.shape[0]
ii = (miss>0) & (miss<0.05)  #less than 5% missingness
mechventcol = reformat3t.columns.tolist().index('mechvent')

for i in range(12, mechventcol): # correct col by col, otherwise it does it wrongly
    if ii[i]==1:
        reformat3[:, i] = fixgaps(reformat3[:, i])

reformat3t[reformat3t.columns[12:mechventcol]] = reformat3[:, 12:mechventcol]

# KNN IMPUTATION - Done on chunks of 10K records.
print('Heparin Cohort -- KNN imputation with K = 1')

reformat3t_cols = reformat3t.columns.tolist()
mechventcol = reformat3t_cols.index('mechvent')
ref = np.copy(reformat3[:,12:mechventcol])  #columns of interest

bar_knn = pyprind.ProgBar(len(range(0,reformat3.shape[0],9999)))
for i in range(0,reformat3.shape[0],9999):   # Dataset divided in 10K rows chunks (otherwise too large)
    bar_knn.update()
    ref[i:i+9999,:] = KNN(k=1).fit_transform(ref[i:i+9999,:])

# Copy on the interpolated data
reformat3t[reformat3t_cols[12:mechventcol]] = ref 

reformat4t = reformat3t.copy()  # Make final table copy to compute derived features

In [ ]:
########################################################################
#        COMPUTE SOME DERIVED FEATURES: P/F, Shock Index, SOFA, SIRS...
########################################################################
# Recompute P/F with no missing values...
reformat4t['PaO2_FiO2'] = reformat4t['paO2']/reformat4t['FiO2_1']

# Recompute SHOCK INDEX without NAN and INF
reformat4t['Shock_Index'] = reformat4t['HR']/reformat4t['SysBP']

reformat4t.loc[np.isinf(reformat4t['Shock_Index']), 'Shock_Index'] = np.NaN
d = np.nanmean(reformat4t['Shock_Index'])
reformat4t['Shock_Index'].fillna(d, inplace=True)

shock_idx = reformat4t['Shock_Index'] >= np.quantile(reformat4t['Shock_Index'], 0.999)
reformat4t.loc[shock_idx, 'Shock_Index'] = np.quantile(reformat4t['Shock_Index'], 0.999)

# SOFA - at each timepoint we need (in this order):  
# P/F,  MV,  PLT,  TOT_BILI,  MAP,  NORAD(max),  GCS, CR,  UO
s = reformat4t[['PaO2_FiO2', 'Platelets_count', 'Total_bili', 'MeanBP', 'max_dose_vaso', 'GCS', 'Creatinine', 'output_1hourly']].values
p = np.arange(5)
s1=np.array([s[:, 0]>400, (s[:, 0]>=300) & (s[:, 0]<400), (s[:, 0]>=200) & (s[:, 0]<300), (s[:, 0]>=100) & (s[:, 0]<200), s[:, 0]<100 ])   #count of points for all 6 criteria of sofa
s2=np.array([s[:, 1]>150, (s[:, 1]>=100) & (s[:, 1]<150), (s[:, 1]>=50) & (s[:, 1]<100), (s[:, 1]>=20) & (s[:, 1]<50), s[:, 1]<20 ])
s3=np.array([s[:, 2]<1.2, (s[:, 2]>=1.2) & (s[:, 2]<2), (s[:, 2]>=2) & (s[:, 2]<6), (s[:, 2]>=6) & (s[:, 2]<12), s[:, 2]>12 ])
s4=np.array([s[:, 3]>=70, (s[:, 3]<70) & (s[:, 3]>=65), (s[:, 3]<65), (s[:, 4]>0) & (s[:, 4]<=0.1), s[:, 4]>0.1 ])
s5=np.array([s[:,5]>14, (s[:, 5]>12) & (s[:, 5]<=14), (s[:, 5]>9) & (s[:, 5]<=12), (s[:, 5]>5) & (s[:, 5]<=9), s[:, 5]<=5])
s6=np.array([s[:,6]<1.2, (s[:,6 ]>=1.2) & (s[:, 6]<2), (s[:, 6]>=2) & (s[:, 6]<3.5), ((s[:, 6]>=3.5) & (s[:, 6]<5))|(s[:, 7]<84), (s[:, 6]>5)|(s[:, 7]<34)])

num_columns = reformat4t.shape[1]   # Number of variables in data
newcols_reformat4 = np.zeros((reformat4t.shape[0],7))
for i in range(reformat4t.shape[0]): 
    t = max(p[s1[:,i]], default=0) + max(p[s2[:,i]], default=0) + max(p[s3[:,i]], default=0) + max(p[s4[:,i]], default=0) \
        + max(p[s5[:,i]], default=0) + max(p[s6[:,i]], default=0)  #SUM OF ALL 6 CRITERIA
    if t > 0:
        newcols_reformat4[i,:] = [max(p[s1[:,i]], default=0), max(p[s2[:,i]], default=0), max(p[s3[:,i]], default=0), 
            max(p[s4[:,i]], default=0), max(p[s5[:,i]], default=0), max(p[s6[:,i]], default=0), t]

# SIRS - at each timepoint |  need: temp HR RR PaCO2 WBC 
s = reformat4t[['Temp_C', 'HR', 'RR', 'paCO2', 'WBC_count']].values

s1=np.array([(s[:,0]>=38) | (s[:,0]<=36)])   #count of points for all criteria of SIRS
s2=np.array([s[:,1]>90])
s3=np.array([(s[:,2]>=20) | (s[:,3]<=32)])
s4=np.array([(s[:,4]>=12) | (s[:,4]<4)])
newcols_sirs = (1*s1) + (1*s2) + (1*s3) + (1*s4)

# more IO corrections
reformat4t.loc[reformat4t['heparin_total'] < 0, 'heparin_total'] = 0
reformat4t.loc[reformat4t['heparin_1hourly'] < 0, 'heparin_1hourly'] = 0

# records values
reformat4t['SOFA'] = newcols_reformat4[:,-1]
reformat4t['SIRS'] = newcols_sirs[0]

In [ ]:
a = reformat4t['heparin_1hourly'].values
plt.boxplot(a)

In [ ]:
print(reformat4t.shape)

In [ ]:
len(reformat4t['icustayid'].drop_duplicates())

In [ ]:
########################################################################
#                     CREATE FINAL MIMIC_TABLE
########################################################################
MIMICtable = reformat4t.copy()
MIMICtable.to_csv('./final_result/1hourly_discrete/MIMICtable.csv', index=False)

In [ ]:
MIMICtable = pd.read_csv("./final_result/1hourly_discrete/no_reward_SAH/MIMICtable.csv")

In [ ]:
MIMICtable['reward_PTT'] = MIMICtable['PTT'].apply(lambda x : (2/(1+np.exp(-(x-60)))) - (2/(1+np.exp(-(x-100))))-1)

In [ ]:
MIMICtable.loc[MIMICtable['reward_PTT'].isna(),'reward_PTT']=0

In [ ]:
MIMICtable

In [ ]:
print(len(MIMICtable.loc[MIMICtable['reward_PTT']>0]))
print(len(MIMICtable.loc[MIMICtable['reward_PTT']<0]))
print(len(MIMICtable.loc[MIMICtable['reward_PTT']==0]))

In [ ]:
#################   Convert training data and compute conversion factors    ######################

# all 47 columns of interest + additional meta columns to easier associate trajectories with other patient auxiliary info (eg. notes)
colmeta = ['presumed_onset', 'charttime', 'icustayid']  # Meta-data around patient stay
colbin = ['gender', 'mechvent', 'max_dose_vaso', 're_admission']  # Binary features
# Patient features that will be z-normalize
colnorm = ['age', 'Weight_kg', 'GCS', 'HR', 'SysBP', 'MeanBP', 'DiaBP', 'RR', 'Temp_C', 'FiO2_1',\
        'Potassium', 'Sodium', 'Chloride', 'Glucose', 'Magnesium', 'Calcium', 'Hb', \
        'WBC_count', 'Platelets_count', 'PTT', 'PT', 'Arterial_pH', 'paO2', 'paCO2',\
        'Arterial_BE', 'HCO3', 'Arterial_lactate', 'SOFA', 'SIRS', 'Shock_Index',\
        'PaO2_FiO2']
# Patient features for log normalization: apply log scaling followed by z-scoring.
collog=['SpO2', 'BUN', 'Creatinine', 'SGOT', 'SGPT', 'Total_bili', 'INR',\
        'heparin_total', 'heparin_1hourly', 'output_total', 'output_1hourly']

# find patients who died in ICU during data collection period
icustayidlist = MIMICtable['icustayid']
icuuniqueids = icustayidlist.unique() #list of unique icustayids from MIMIC

MIMICraw = MIMICtable[colmeta + colbin + colnorm + collog]
MIMICraw = MIMICraw.values  # RAW values
MIMICzs = np.hstack([MIMICtable[colmeta].values, MIMICtable[colbin].values-0.5, stats.zscore(MIMICtable[colnorm].values), 
    stats.zscore(np.log(0.1 + MIMICtable[collog].values))])

MIMICzs[:,5] = np.log(MIMICzs[:, 5] + 0.6)  # MAX DOSE NORAD(vaso)
MIMICzs[:,46] = 2*MIMICzs[:,46]  # Increase weight of this variable


In [ ]:
a = MIMICtable['heparin_1hourly'].values  # heparin IV fluid

In [ ]:
a

In [ ]:
len(a[a==0])

In [ ]:
a = a[a<6000]

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.transforms import blended_transform_factory

# Parameters.
thresholds = [100, 752, 1000, 1350, 1851]   # Position of the vertical line.
quant_labels = ["Q0", "Q20", "Q40", "Q60", "Q80"]  # Text to display.
cutoff = 3000                                # Boundary of the omitted x-axis interval.
bins_left = 30
dpi = 300

# Bar-outline style.
bar_edgecolor = "black"
bar_edgewidth = 0.8

# Y-axis break threshold; above this value, omit the middle interval and show only the maximum in the upper panel.
y_cutoff = 5000

# Fixed single y-axis tick to display in the upper panel.
top_tick_value = 11000

# Data: assume that array a already exists.
# a = np.array([...])

# Style settings.
mpl.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16
})
fmt_thousand = FuncFormatter(lambda x, pos: f"{int(x):,}" if x >= 1 else f"{x:g}")

a = np.asarray(a)
a = a[~np.isnan(a)]
max_a = float(np.max(a))

left = a[a < cutoff]
right = a[a >= cutoff]

# 2x2 layout: thin upper panels show only the y-maximum; lower panels show 0 to y_cutoff; left/right columns split at cutoff.
fig, axs = plt.subplots(
    2, 2,
    figsize=(9.2, 5.6),
    sharex='col',  # Share the x-axis between rows.
    gridspec_kw={"height_ratios": [1, 4], "width_ratios": [4, 1]}
)
axTopL, axTopR = axs[0]
axBotL, axBotR = axs[1]

# Lower-left panel: 0 to cutoff.
nBL, binsL, patchesBL = axBotL.hist(
    left, bins=bins_left, rwidth=1,
    edgecolor=bar_edgecolor, linewidth=bar_edgewidth
)
axBotL.set_xlim(0, cutoff)
axBotL.xaxis.set_major_formatter(fmt_thousand)
axBotL.set_xticks(np.arange(0, cutoff + 1, 500))
axBotL.set_xlabel("Heparin Dose (units)")
axBotL.set_ylabel("Count")
axBotL.grid(alpha=0.3)

# Lower-right panel: values above cutoff.
if right.size > 0:
    nbins_right = max(1, min(10, int(np.sqrt(right.size))))
    nBR, binsR, patchesBR = axBotR.hist(
        right, bins=nbins_right, rwidth=0.98,
        edgecolor=bar_edgecolor, linewidth=bar_edgewidth
    )
    rmin = float(np.min(right))
    pad = 0.02 * max(1.0, (max_a - rmin))
    # Show only the maximum x-axis tick, such as 6000, in the right panel.
    max_xtick = int(np.ceil(max_a / 1000.0) * 1000)
    eps_x = 0.02 * max(1.0, (max_xtick - rmin))
    axBotR.set_xlim(rmin - pad, max_xtick + eps_x)
    axBotR.set_xticks([max_xtick])
    axBotR.xaxis.set_major_formatter(fmt_thousand)
    axBotR.grid(alpha=0.3)
else:
    nBR = np.array([0.0])
    axBotR.axis("off")

# Hide the y-axis ticks and labels in the right column.
for ax in (axTopR, axBotR):
    ax.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)

# Compute the maximum y-axis value.
y_max = float(max(np.max(nBL) if nBL.size else 0.0,
                  np.max(nBR) if nBR.size else 0.0))

# Draw the same histogram in the thin upper panels without text labels.
axTopL.hist(
    left, bins=bins_left, rwidth=1,
    edgecolor=bar_edgecolor, linewidth=bar_edgewidth
)
if right.size > 0:
    axTopR.hist(
        right, bins=max(1, min(10, int(np.sqrt(right.size)))),
        rwidth=0.98, edgecolor=bar_edgecolor, linewidth=bar_edgewidth
    )
else:
    axTopR.axis("off")

# Draw the vertical lines and text labels only in the lower panels.
first = True
for t, qtxt in zip(thresholds, quant_labels):
    if t < cutoff:
        # Lower-left panel: vertical line and text.
        axBotL.axvline(t, color="red", linestyle="--", linewidth=1,
                       label="Quantile Thresholds" if first else None)
        transB = blended_transform_factory(axBotL.transData, axBotL.transAxes)
        axBotL.text(t, 0.97, f"{qtxt}", transform=transB,
                    va="top", ha="center", fontsize=14, color="red",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))
        # Upper-left panel: retain the vertical line without text.
        axTopL.axvline(t, color="red", linestyle="--", linewidth=1)
        first = False
    else:
        if right.size > 0:
            # Lower-right panel: vertical line and text.
            axBotR.axvline(t, color="red", linestyle="--", linewidth=1,
                           label="Quantile Thresholds" if first else None)
            transB = blended_transform_factory(axBotR.transData, axBotR.transAxes)
            axBotR.text(t, 0.97, f"{qtxt}", transform=transB, rotation=90,
                        va="top", ha="center", fontsize=14, color="red",
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))
            # Upper-right panel: retain the vertical line without text.
            axTopR.axvline(t, color="red", linestyle="--", linewidth=1)
            first = False

# Configure the broken y-axis.
if y_max >= y_cutoff:
    # Lower panels: 0 to y_cutoff.
    axBotL.set_ylim(0, y_cutoff)
    if right.size > 0:
        axBotR.set_ylim(0, y_cutoff)

    # Upper panels: display only the fixed tick at 11000.
    # Automatically choose a narrow range containing both 11000 and the actual y_max.
    band_ref = max(y_max, top_tick_value)
    dy = max(1.0, 0.02 * band_ref)  # Thickness of the narrow range.
    top_ymin = min(top_tick_value, y_max) - dy
    top_ymax = max(top_tick_value, y_max) + dy

    axTopL.set_ylim(top_ymin, top_ymax)
    axTopL.set_yticks([top_tick_value])          # Display only 11000.
    axTopL.yaxis.set_major_formatter(fmt_thousand)
    axTopL.tick_params(labelbottom=False)

    if right.size > 0:
        axTopR.set_ylim(top_ymin, top_ymax)
        # Keep the y-axis ticks and labels hidden in the upper-right panel.
        axTopR.tick_params(labelbottom=False)
else:
    axTopL.axis("off")
    if right.size > 0:
        axTopR.axis("off")

# Helper functions for drawing axis breaks.
d = .015

def add_x_break_marks(ax_left, ax_right):
    kwargs = dict(transform=ax_left.transAxes, color='k', clip_on=False, linewidth=1.2)
    ax_left.plot((1 - d, 1 + d), (-d, +d), **kwargs)
    ax_left.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
    kwargs.update(transform=ax_right.transAxes)
    ax_right.plot((-d, +d), (-d, +d), **kwargs)
    ax_right.plot((-d, +d), (1 - d, 1 + d), **kwargs)

def add_y_break_marks(ax_top, ax_bottom):
    kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, linewidth=1.2)
    ax_top.plot((-d, +d), (-d, +d), **kwargs)                 # Lower left.
    ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)           # Lower right.
    kwargs.update(transform=ax_bottom.transAxes)
    ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)        # Upper left.
    ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)  # Upper right.

# Draw x-axis breaks between the left and right columns in both rows.
if right.size > 0:
    add_x_break_marks(axBotL, axBotR)
    if axTopL.axes.get_visible() and axTopR.axes.get_visible():
        add_x_break_marks(axTopL, axTopR)

# Draw y-axis breaks between the upper and lower rows in both columns.
if axTopL.axes.get_visible():
    add_y_break_marks(axTopL, axBotL)
if right.size > 0 and axTopR.axes.get_visible():
    add_y_break_marks(axTopR, axBotR)

# Display one legend in the lower-left panel.
if not first:
    axBotL.legend(
        fontsize=16,
        loc="upper right",
        bbox_to_anchor=(1.0, 0.95),
        borderaxespad=0.8
    )

fig.tight_layout()
fig.savefig("./action_distribution_mimic3.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
no_zero = a[a>0]
qu1 = np.quantile(no_zero, 0.2)
qu2 = np.quantile(no_zero, 0.4)
qu3 = np.quantile(no_zero, 0.6)
qu4 = np.quantile(no_zero, 0.8)

In [ ]:
print(int(qu1), int(qu2), int(qu3), int(qu4))

In [ ]:
dis_a = []

for i in a:
    if i ==0:
        dis_a.append(0)
    elif 0<i<=qu1:
        dis_a.append(1)
    elif qu1<i<=qu2:
        dis_a.append(2)
    elif qu2<i<=qu3:
        dis_a.append(3)
    elif qu3<i<=qu4:
        dis_a.append(4)
    else:
        dis_a.append(5)

In [ ]:
from collections import Counter
Counter(dis_a)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

Y = [9817,9367,9369,9426,9293,9460]
X = ["action1","action2","action3","action4","action5","action6"]

fig, ax = plt.subplots(1)

ax.bar(X, Y, color='tab:blue', alpha=0.9)
plt.ylabel("Count")
plt.xlabel("actions")
ax.grid(alpha=0.3)
ax.patch.set_facecolor("gray")
ax.patch.set_alpha(0.1)
plt.savefig("./final_result/action_count.png", dpi=600)
plt.show()


In [ ]:
# Process MIMICzs into trajectory data for later use
MIMICzs             = pd.DataFrame(MIMICzs, columns=colmeta+colbin+colnorm+collog)
meta_df             = pd.DataFrame(MIMICzs[colmeta].values, columns = colmeta)
ob_df               = pd.DataFrame(MIMICzs[colbin+colnorm+collog].values, columns=colbin+colnorm+collog)
ac_df               = pd.DataFrame(dis_a, columns=['action'])
raw_data_df         = MIMICtable.copy()
num_actions         = 6
outcome_key         = 'died_within_48h_of_out_time' #Should be either 'died_within_48h_of_out_time' or ''mortality_90d''
meta_cols           = meta_df.columns.tolist()
ob_cols             = ob_df.columns.tolist()
raw_data_df['traj'] = (raw_data_df['bloc'] == 1).cumsum().values
meta_df['traj']     = (raw_data_df['bloc'] == 1).cumsum().values
ob_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
ac_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
trajectories        = raw_data_df['traj'].unique()
data = {}
data['meta_cols'] = meta_cols
data['obs_cols'] = ob_cols
data['traj'] = {}

In [ ]:
print('Heparin Cohort -- Making trajectory data')
bar = pyprind.ProgBar(len(trajectories))
for i in trajectories:
    bar.update()
    data['traj'][i] = {}
    data['traj'][i]['meta'] = meta_df[meta_df['traj']==i][meta_cols].values.T
    data['traj'][i]['obs'] = ob_df[ob_df['traj'] == i][ob_cols].values.T
    data['traj'][i]['actions'] = ac_df[ac_df['traj'] == i]['action'].values.astype(np.int32)
    data['traj'][i]['outcome'] = raw_data_df[raw_data_df['traj'] == i][outcome_key].values[0] 
    data['traj'][i]['rewards'] = np.zeros(len(data['traj'][i]['actions']))

In [ ]:
print('Heparin Cohort -- Making final output file')
col_names = ['traj', 'step']
col_names.extend(['m:'+ i for i in data['meta_cols']])
col_names.extend(['o:'+ i for i in data['obs_cols']])
col_names.append('a:action')
col_names.append('r:reward')
all_data = []
bar = pyprind.ProgBar(len(data['traj'].keys()))
for i in data['traj'].keys():
    bar.update()
    for ctr in range(data['traj'][i]['actions'].shape[0]):
        all_data.append([])
        all_data[-1].append(i)
        all_data[-1].append(ctr)
        for m_index in range(data['traj'][i]['meta'].shape[0]):
            all_data[-1].append(data['traj'][i]['meta'][m_index, ctr])
        for o_index in range(data['traj'][i]['obs'].shape[0]):
            all_data[-1].append(data['traj'][i]['obs'][o_index, ctr])
        all_data[-1].append(data['traj'][i]['actions'][ctr])
        all_data[-1].append(data['traj'][i]['rewards'][ctr])
df = pd.DataFrame(all_data, columns=col_names)


In [ ]:
np.min(df['o:heparin_1hourly'])

In [ ]:
np.max(df['o:heparin_1hourly'].values)

In [ ]:
df['PTT_reward']=MIMICtable['reward_PTT'].values # Use the reward values without scaling.

In [ ]:
df.drop(columns=['r:reward',"o:max_dose_vaso","o:mechvent"], inplace=True)
df.to_csv('./final_result/1hourly_discrete/heparin_final_data_withTimes.csv', index=False)

# o:PTT 는 interpolation을 모두 했고 aPTT_reward에는 전혀 적용안함 real 데이터임.

In [ ]:
raw_df              = pd.DataFrame(MIMICraw, columns=colmeta+colbin+colnorm+collog)
meta_df             = raw_df[colmeta]
meta_df             = pd.DataFrame(meta_df.values, columns=colmeta)
ob_df               = raw_df[colbin+colnorm+collog]
ob_df               = pd.DataFrame(ob_df.values, columns=colbin+colnorm+collog)
ac_df               = pd.DataFrame(dis_a, columns=['action'])
raw_data_df         = MIMICtable.copy()
num_actions         = 6
outcome_key         = 'died_within_48h_of_out_time' #Should be either 'died_within_48h_of_out_time' or ''mortality_90d''
meta_cols           = meta_df.columns.tolist()
ob_cols             = ob_df.columns.tolist()
raw_data_df['traj'] = (raw_data_df['bloc'] == 1).cumsum().values
meta_df['traj']     = (raw_data_df['bloc'] == 1).cumsum().values
ob_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
ac_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
trajectories        = raw_data_df['traj'].unique()
data = {}
data['meta_cols'] = meta_cols
data['obs_cols'] = ob_cols
data['traj'] = {}
print('Heparin Cohort -- Making RAW trajectory data')
bar = pyprind.ProgBar(len(trajectories))
for i in trajectories:
    bar.update()
    data['traj'][i] = {}
    data['traj'][i]['meta'] = meta_df[meta_df['traj']==i][meta_cols].values.T
    data['traj'][i]['obs'] = ob_df[ob_df['traj'] == i][ob_cols].values.T
    data['traj'][i]['actions'] = ac_df[ac_df['traj'] == i]['action'].values.astype(np.int32)
    data['traj'][i]['outcome'] = raw_data_df[raw_data_df['traj'] == i][outcome_key].values[0] 
    data['traj'][i]['rewards'] = np.zeros(len(data['traj'][i]['actions']))

print('Heparin Cohort -- Making final RAW output file')
col_names = ['traj', 'step']
col_names.extend(['m:'+ i for i in data['meta_cols']])
col_names.extend(['o:'+ i for i in data['obs_cols']])
col_names.append('a:action')
col_names.append('r:reward')
all_data = []
bar = pyprind.ProgBar(len(data['traj'].keys()))
for i in data['traj'].keys():
    bar.update()
    for ctr in range(data['traj'][i]['actions'].shape[0]):
        all_data.append([])
        all_data[-1].append(i)
        all_data[-1].append(ctr)
        for m_index in range(data['traj'][i]['meta'].shape[0]):
            all_data[-1].append(data['traj'][i]['meta'][m_index, ctr])            
        for o_index in range(data['traj'][i]['obs'].shape[0]):
            all_data[-1].append(data['traj'][i]['obs'][o_index, ctr])
        all_data[-1].append(data['traj'][i]['actions'][ctr])
        all_data[-1].append(data['traj'][i]['rewards'][ctr])
df = pd.DataFrame(all_data, columns=col_names)


In [ ]:
df

In [ ]:
df['PTT_reward']=MIMICtable['reward_PTT'].values # Use the reward values without scaling.
df.drop(columns=['r:reward',"o:max_dose_vaso","o:mechvent"], inplace=True)
df.to_csv('./final_result/1hourly_discrete/heparin_final_data_RAW_withTimes.csv', index=False)

In [ ]:
df

## 통계 확인

In [ ]:
df =pd.read_csv('./final_result/1hourly_discrete/heparin_final_data_RAW_withTimes.csv')